# 🎯 [K13] 군 교육 콘텐츠 추천 AI - 장병 맞춤 교육 콘텐츠 추천 구현
### Google Colab 실습 환경 (교육훈련·역량개발 / Level 2)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/edunlab/milicontent2/blob/main/uploads/K13/K13_solution.ipynb)

---

## 📌 1. 프로젝트 개요 및 배경

| 항목 | 내용 |
| :--- | :--- |
| **과정 분류** | 교육훈련·역량개발 (Level 2) |
| **학습 소요 시간** | 10개 미션 (5일 / 총 20.0시간) |
| **환경 사양** | Python 3.10+, numpy, pandas, matplotlib, scipy, scikit-learn |

### 📖 프로젝트 목표
장병은 관심 분야, 계급, 남은 복무 기간이 저마다 달라 수많은 교육 콘텐츠 중 자신에게 맞는 것을 고르기 어렵고, 모두에게 같은 인기 과정만 안내하면 개인 필요를 놓치게 됩니다. 이 프로젝트에서는 파이썬으로 가상 교육 콘텐츠 40개와 가상 장병 60명의 수강 이력·평점을 구조화하고, 인기도 기준 추천, 설명 유사도(TF-IDF) 기반 추천, 유사 장병 협업 필터링과 행렬 분해, 하이브리드 추천을 단계별로 구현합니다. Precision@5·Recall@5·커버리지로 추천 성능을 평가하고, 이력이 없는 신규 전입 장병 대응, 추천 이유 설명, 같은 주제만 추천되는지 확인하는 편향·다양성 점검까지 수행하는 '교육 콘텐츠 추천 도우미'를 완성합니다. 추천은 어디까지나 제안이며 교육 이수 여부는 본인과 지휘관이 결정한다는 원칙을 함께 익힙니다.

> **⚠️ [교육용 가상 데이터셋 원칙]**
> - 본 실습의 모든 데이터(공지, 규정, 일정 등)는 보안 규정을 준수하여 **코드 내 난수 및 가상 시나리오로 생성**됩니다.
> - 실제 부대 작전, 군사 기밀 및 개인 식별 정보는 일체 포함되지 않습니다.

---



In [ ]:
# ============================================================================
# [Google Colab 환경 설정] 필수 라이브러리 설치 및 임포트
# ============================================================================
!pip install -q numpy pandas matplotlib scipy scikit-learn

import os
import sys
import time
import math
import json
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# 그래프 영문 폰트 기본 설정 (환경 독립성 보장)
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

print(f"실행 환경 준비 완료 (Python {sys.version.split()[0]})")


---

## 🎯 미션 01. 교육 데이터 구조화하기
**소요시간**: 약 2.0시간

### 📋 미션 배경 및 수행 지침
#### 미션 배경 및 당위성

실제 부대 실무 현장에서는 불명확한 정보 전달이나 수동 업무 처리로 인해 불필요한 행정 소요가 발생하고 주요 공지·규정 확인이 지연되는 문제가 발생하기 쉽습니다. 따라서 **가상 교육 콘텐츠 40개와 장병 60명의 수강 이력·평점을 표로 구조화하고 형식을 검사합니다.** 과업을 체계적으로 수행하여 현장 업무 효율성을 극대화하고 정보 누락으로 인한 혼선을 사전에 방지해야 합니다.

이 문제를 해결하기 위해 본 미션에서는 **콘텐츠(주제·난이도·시간·설명), 장병(계급·관심 주제), 평점 표를 만들고 주제별 개수와 평점 분포, 비어 있는 칸의 비율(희소도)을 확인하고 유효성을 점검하는** 실습을 단계별로 수행합니다.

학습자는 실습가이드에 제시된 총 4개 세부 활동을 순차적으로 진행하며 핵심 산출물을 도출하고 검증 기준을 달성합니다.

**이번 미션 결과물:** `데이터 표 3종, 형식 검사 함수와 시험 결과, 학습·평가 분리 결과` 완성

### 🔍 평가 및 검증 기준 (PASS 조건)
#### 미션 판정

**평가 구분:** 필수

**핵심 평가 기준:** [필수] 데이터 구조화 — 추천에 쓸 데이터를 일관된 표로 정리하고 형식 오류를 잡아내는 검사와 평가용 분리를 만들었는가? (PASS / FAIL)

#### 미션 PASS 조건

- 요구 산출물(데이터 표 3종, 형식 검사 함수와 시험 결과, 학습·평가 분리 결과)의 완성도 충족


- 원문 대조 점검을 통한 사실 검증 및 정확성 확보


- 교육용 가상 데이터 활용 및 군사보안 수칙 엄수


#### 채점자 판정 메모

> 단순 결과물 생성 여부뿐만 아니라 AI 결과물의 오류를 직접 찾아내고 바로잡았는지 대조표를 확인한다.



In [ ]:
"""가상 교육 콘텐츠 40개와 장병 60명의 수강 이력·평점을 표로 구조화하고, 형식 검사와 희소도(비어 있는 정도)를 확인한다."""
import numpy as np
import pandas as pd

# ---------------------------------------------------------------------------
# [교육용가상자료] 가상 한빛대대 교육 콘텐츠 40개와 가상 장병 60명 (실제 부대·인원과 무관)
# ---------------------------------------------------------------------------
TOPICS = ["AI 기초", "데이터", "보안", "안전", "리더십", "체력", "자격증 대비", "소통·정서"]
LEVELS = {1: "기초", 2: "중급", 3: "심화"}
COURSE_ROWS = [
    ("C01", "인공지능 첫걸음", "AI 기초", 1, 2, "인공지능이 무엇인지 사례로 알아보고 일상과 업무에서 쓰이는 방식을 배웁니다"),
    ("C02", "생성형 AI 안전하게 쓰기", "AI 기초", 1, 1, "생성형 AI에 질문하는 방법과 가상 자료만 입력하는 보안 수칙, 답변 검증 방법을 배웁니다"),
    ("C03", "머신러닝 맛보기", "AI 기초", 2, 3, "데이터로 규칙을 학습하는 머신러닝의 원리와 분류 예측 실습을 해 봅니다"),
    ("C04", "AI 프롬프트 작성 실습", "AI 기초", 2, 2, "목적에 맞는 프롬프트를 작성하고 결과를 비교하며 개선하는 실습을 합니다"),
    ("C05", "AI 윤리와 책임", "AI 기초", 2, 1, "AI 결과의 한계와 편향, 최종 판단은 사람이 한다는 원칙을 사례로 배웁니다"),
    ("C06", "엑셀 데이터 정리 기초", "데이터", 1, 2, "표 데이터를 정리하고 합계와 평균을 구하며 오류를 찾는 방법을 배웁니다"),
    ("C07", "데이터 시각화 입문", "데이터", 1, 2, "막대그래프와 꺾은선 그래프로 데이터를 읽기 쉽게 보여 주는 방법을 배웁니다"),
    ("C08", "파이썬 데이터 분석 기초", "데이터", 2, 4, "파이썬으로 표 데이터를 읽고 통계를 계산하며 그래프를 그리는 실습을 합니다"),
    ("C09", "통계로 읽는 부대 생활", "데이터", 2, 2, "평균과 분포, 상관관계로 생활 데이터를 해석하는 방법을 사례로 배웁니다"),
    ("C10", "데이터 품질 점검 실습", "데이터", 3, 3, "결측과 중복, 형식 오류를 찾아 고치고 데이터 신뢰도를 점검하는 방법을 실습합니다"),
    ("C11", "정보보안 기본 수칙", "보안", 1, 1, "비밀번호 관리와 외부 저장매체 사용 금지, 촬영 금지 등 보안 수칙을 배웁니다"),
    ("C12", "사이버 위협 알아보기", "보안", 1, 2, "악성 메일과 피싱 사례를 보고 의심스러운 연락에 대처하는 방법을 배웁니다"),
    ("C13", "계정과 비밀번호 관리", "보안", 2, 1, "안전한 비밀번호 만들기와 계정 점검 방법을 실습하며 보안 습관을 기릅니다"),
    ("C14", "보안 사고 대응 절차", "보안", 2, 2, "보안 사고가 의심될 때 보고하는 절차와 지휘계통 보고 요령을 사례로 배웁니다"),
    ("C15", "네트워크 보안 이해", "보안", 3, 4, "네트워크 구조와 접근 통제의 원리를 배우고 취약점 점검 개념을 이해합니다"),
    ("C16", "생활관 안전사고 예방", "안전", 1, 1, "전열기 사용과 결빙, 낙상 등 생활관에서 일어나는 안전사고 예방 수칙을 배웁니다"),
    ("C17", "응급처치 기본", "안전", 1, 2, "지혈과 심폐소생술 등 응급처치 기본 동작을 실습하고 보고 요령을 익힙니다"),
    ("C18", "화재 예방과 대피", "안전", 2, 2, "화재 원인과 초기 진화, 대피 경로 확인 방법을 훈련 사례로 배웁니다"),
    ("C19", "동절기 안전 관리", "안전", 2, 1, "한파와 결빙에 대비한 방한 수칙과 동상 예방, 안전 점검 방법을 배웁니다"),
    ("C20", "안전 리더 과정", "안전", 3, 3, "분대 단위 안전 점검표를 만들고 위험 요인을 찾아 개선하는 방법을 실습합니다"),
    ("C21", "분대장 리더십 기초", "리더십", 1, 2, "분대원을 이끄는 기본 자세와 지시, 보고 요령을 사례로 배웁니다"),
    ("C22", "팀 목표 세우기", "리더십", 1, 1, "팀의 목표를 정하고 역할을 나누며 진행 상황을 점검하는 방법을 배웁니다"),
    ("C23", "갈등 조정과 중재", "리더십", 2, 2, "생활관에서 생기는 갈등을 듣고 조정하는 대화 방법을 역할극으로 연습합니다"),
    ("C24", "멘토링과 후임 지도", "리더십", 2, 2, "후임을 지도하고 격려하는 멘토링 방법과 피드백 요령을 사례로 배웁니다"),
    ("C25", "지휘 판단과 의사결정", "리더십", 3, 3, "정보가 부족할 때 근거를 모아 결정하고 책임 있게 보고하는 방법을 사례로 배웁니다"),
    ("C26", "기초 체력 향상 프로그램", "체력", 1, 2, "팔굽혀펴기와 윗몸일으키기, 달리기 기록을 단계별로 올리는 체력 훈련을 합니다"),
    ("C27", "스트레칭과 부상 예방", "체력", 1, 1, "운동 전후 스트레칭과 올바른 자세로 근육 부상을 예방하는 방법을 배웁니다"),
    ("C28", "체력 측정 대비 훈련", "체력", 2, 2, "체력 측정 종목별 요령과 기록 향상 훈련 계획을 세우고 실천합니다"),
    ("C29", "영양과 회복 관리", "체력", 2, 1, "훈련 후 영양 섭취와 수면, 회복 관리로 체력을 유지하는 방법을 배웁니다"),
    ("C30", "전투 체력 심화 과정", "체력", 3, 3, "장거리 달리기와 근지구력을 높이는 심화 체력 훈련과 기록 관리 방법을 배웁니다"),
    ("C31", "컴퓨터활용능력 대비", "자격증 대비", 1, 4, "표 계산과 문서 작성 등 컴퓨터활용능력 시험 문제 풀이와 자격증 대비 요령을 익힙니다"),
    ("C32", "한국사능력 대비", "자격증 대비", 1, 3, "한국사 흐름을 정리하고 기출 문제를 풀며 자격증 시험을 대비합니다"),
    ("C33", "정보처리 기초 대비", "자격증 대비", 2, 4, "자료구조와 알고리즘, 데이터베이스 기초를 정리하고 자격증 시험 문제를 풉니다"),
    ("C34", "데이터 분석 자격 대비", "자격증 대비", 2, 4, "통계와 데이터 처리 개념을 정리하고 데이터 분석 자격증 시험을 대비합니다"),
    ("C35", "전기·정비 기능사 대비", "자격증 대비", 3, 4, "전기 회로와 정비 이론을 정리하고 실기 요령을 익혀 기능사 자격증을 대비합니다"),
    ("C36", "말하기와 발표 기초", "소통·정서", 1, 1, "핵심을 먼저 말하는 보고 방식과 발표 자세를 연습하며 소통 능력을 기릅니다"),
    ("C37", "스트레스 관리와 마음 건강", "소통·정서", 1, 2, "스트레스 신호를 알아차리고 상담 창구를 이용하는 방법 등 마음 건강 관리를 배웁니다"),
    ("C38", "경청과 공감 대화", "소통·정서", 2, 2, "상대의 말을 끝까지 듣고 공감하며 답하는 대화 방법을 짝 연습으로 익힙니다"),
    ("C39", "전역 후 진로 설계", "소통·정서", 2, 2, "전역 후 진로를 정하고 자기개발 계획과 자격증 준비 일정을 세우는 방법을 배웁니다"),
    ("C40", "부대 보고서 글쓰기", "소통·정서", 3, 3, "읽는 사람을 생각해 근거와 결론이 분명한 보고서와 공문을 쓰는 방법을 실습합니다"),
]
courses = pd.DataFrame(COURSE_ROWS, columns=["id", "title", "topic", "level", "hours", "desc"])
CID = list(courses["id"])
RANKS = ["이병", "일병", "상병", "병장"]

# 가상 장병 60명: 계급, 관심 주제 2개, 수강 이력과 평점(1~5)을 난수로 만든다
rng = np.random.default_rng(13)
item_pop = rng.lognormal(0, 0.6, len(courses))
rows, soldier_rows = [], []
for i in range(60):
    sid = f"S{i + 1:02d}"
    rank = RANKS[min(i // 15, 3)]
    interest = list(rng.choice(TOPICS, 2, replace=False))
    max_level = 1 + (RANKS.index(rank) >= 1) + (RANKS.index(rank) >= 3)
    w = np.array([item_pop[j] * (1 + 7 * (courses.topic[j] in interest)) * (1.0 if courses.level[j] <= max_level else 0.3)
                  for j in range(len(courses))])
    n_take = int(rng.integers(7, 14))
    picked = rng.choice(len(courses), n_take, replace=False, p=w / w.sum())
    for j in picked:
        score = 2.9 + 1.3 * (courses.topic[j] in interest) + rng.normal(0, 0.7)
        rows.append((sid, CID[j], int(np.clip(round(score), 1, 5))))
    soldier_rows.append((sid, rank, interest[0], interest[1]))
soldiers = pd.DataFrame(soldier_rows, columns=["id", "rank", "interest1", "interest2"])
ratings = pd.DataFrame(rows, columns=["soldier", "course", "rating"])
SID = list(soldiers["id"])

# 평가용 분리: 평점 4 이상인 수강 중 장병마다 최대 2개를 숨겨 두고(test), 나머지로 추천 모델을 만든다(train)
test_idx = []
for sid, g in ratings[ratings.rating >= 4].groupby("soldier"):
    test_idx += list(rng.choice(g.index, min(2, len(g)), replace=False))
test = ratings.loc[sorted(test_idx)].reset_index(drop=True)
train = ratings.drop(index=test_idx).reset_index(drop=True)
R = train.pivot(index="soldier", columns="course", values="rating").reindex(index=SID, columns=CID)
M = (~R.isna()).values.astype(float)        # 수강 여부(1/0)
RV = R.fillna(0).values                      # 평점 행렬(미수강은 0)
relevant = test.groupby("soldier")["course"].apply(set).to_dict()


def check_ratings(df, course_ids, soldier_ids):
    """평점 표의 오류(없는 장병·콘텐츠, 범위 밖 평점, 중복 수강)를 찾아 문장으로 돌려준다."""
    errs = []
    errs += [f"없는 장병 {s}" for s in sorted(set(df["soldier"]) - set(soldier_ids))]
    errs += [f"없는 콘텐츠 {c}" for c in sorted(set(df["course"]) - set(course_ids))]
    errs += [f"평점 범위 오류({r})" for r in df.loc[~df["rating"].between(1, 5), "rating"]]
    if df.duplicated(["soldier", "course"]).any():
        errs.append("같은 콘텐츠 중복 수강 기록")
    return errs


print("[콘텐츠 표 일부]")
print(courses[["id", "title", "topic", "level", "hours"]].head(8).to_string(index=False))
print("\n주제별 콘텐츠 수:", courses["topic"].value_counts().to_dict())
print("난이도별 콘텐츠 수:", {LEVELS[k]: v for k, v in courses["level"].value_counts().sort_index().items()})
print(f"평균 교육 시간 {courses['hours'].mean():.1f}시간")

print("\n[장병 표 일부]")
print(soldiers.head(5).to_string(index=False))
print("계급별 인원:", soldiers["rank"].value_counts().reindex(RANKS).to_dict())

print(f"\n수강 기록 {len(ratings)}건, 장병 1인당 평균 {len(ratings) / len(soldiers):.1f}개 수강")
print("평점 분포:", ratings["rating"].value_counts().sort_index().to_dict())
full = ratings.pivot(index="soldier", columns="course", values="rating")
sparsity = full.isna().mean().mean()
print(f"장병x콘텐츠 행렬 {full.shape}, 비어 있는 칸 {sparsity:.0%} (대부분의 콘텐츠를 아직 안 들음)")
print(f"가장 적게 수강된 콘텐츠: {ratings['course'].value_counts().idxmin()} ({ratings['course'].value_counts().min()}건)")

print("\n[형식 검사]")
print("정상 데이터 오류:", check_ratings(ratings, CID, SID))
bad = pd.DataFrame([("S99", "C01", 4), ("S01", "C99", 3), ("S02", "C02", 7), ("S03", "C03", 4), ("S03", "C03", 5)],
                   columns=["soldier", "course", "rating"])
print("오류 주입 시험:", check_ratings(bad, CID, SID))

print(f"\n[평가용 분리] 학습 {len(train)}건, 평가용(평점 4 이상 중 숨긴 기록) {len(test)}건, 평가 대상 장병 {len(relevant)}명")


---

## 🎯 미션 02. 인기도 기준 추천 만들기
**소요시간**: 약 2.0시간

### 📋 미션 배경 및 수행 지침
#### 미션 배경 및 당위성

실제 부대 실무 현장에서는 불명확한 정보 전달이나 수동 업무 처리로 인해 불필요한 행정 소요가 발생하고 주요 공지·규정 확인이 지연되는 문제가 발생하기 쉽습니다. 따라서 **인기도 기준(Baseline) 추천을 만들고 수강 수가 적은 콘텐츠의 평균 평점을 보정합니다.** 과업을 체계적으로 수행하여 현장 업무 효율성을 극대화하고 정보 누락으로 인한 혼선을 사전에 방지해야 합니다.

이 문제를 해결하기 위해 본 미션에서는 **콘텐츠별 평균 평점과 수강 수를 구하고, 수강 수가 적을수록 전체 평균에 가깝게 보정한 점수로 순위를 매깁니다 과정을 진행하는** 실습을 단계별로 수행합니다.

학습자는 실습가이드에 제시된 총 4개 세부 활동을 순차적으로 진행하며 핵심 산출물을 도출하고 검증 기준을 달성합니다.

**이번 미션 결과물:** `인기도 추천 함수, 보정 전후 순위표, Baseline 성능` 완성

### 🔍 평가 및 검증 기준 (PASS 조건)
#### 미션 판정

**평가 구분:** 필수

**핵심 평가 기준:** [필수] Baseline 구현 — 인기도 추천을 구현하고 보정의 이유와 기준선 성능을 확인했는가? (PASS / FAIL)

#### 미션 PASS 조건

- 요구 산출물(인기도 추천 함수, 보정 전후 순위표, Baseline 성능)의 완성도 충족


- 원문 대조 점검을 통한 사실 검증 및 정확성 확보


- 교육용 가상 데이터 활용 및 군사보안 수칙 엄수


#### 채점자 판정 메모

> 단순 결과물 생성 여부뿐만 아니라 AI 결과물의 오류를 직접 찾아내고 바로잡았는지 대조표를 확인한다.



In [ ]:
"""인기도 기준(Baseline) 추천을 만들고, 수강 수가 적은 콘텐츠의 평균 평점을 보정하는 이유를 확인한다."""
import numpy as np
import pandas as pd

# ---------------------------------------------------------------------------
# [교육용가상자료] 가상 한빛대대 교육 콘텐츠 40개와 가상 장병 60명 (실제 부대·인원과 무관)
# ---------------------------------------------------------------------------
TOPICS = ["AI 기초", "데이터", "보안", "안전", "리더십", "체력", "자격증 대비", "소통·정서"]
LEVELS = {1: "기초", 2: "중급", 3: "심화"}
COURSE_ROWS = [
    ("C01", "인공지능 첫걸음", "AI 기초", 1, 2, "인공지능이 무엇인지 사례로 알아보고 일상과 업무에서 쓰이는 방식을 배웁니다"),
    ("C02", "생성형 AI 안전하게 쓰기", "AI 기초", 1, 1, "생성형 AI에 질문하는 방법과 가상 자료만 입력하는 보안 수칙, 답변 검증 방법을 배웁니다"),
    ("C03", "머신러닝 맛보기", "AI 기초", 2, 3, "데이터로 규칙을 학습하는 머신러닝의 원리와 분류 예측 실습을 해 봅니다"),
    ("C04", "AI 프롬프트 작성 실습", "AI 기초", 2, 2, "목적에 맞는 프롬프트를 작성하고 결과를 비교하며 개선하는 실습을 합니다"),
    ("C05", "AI 윤리와 책임", "AI 기초", 2, 1, "AI 결과의 한계와 편향, 최종 판단은 사람이 한다는 원칙을 사례로 배웁니다"),
    ("C06", "엑셀 데이터 정리 기초", "데이터", 1, 2, "표 데이터를 정리하고 합계와 평균을 구하며 오류를 찾는 방법을 배웁니다"),
    ("C07", "데이터 시각화 입문", "데이터", 1, 2, "막대그래프와 꺾은선 그래프로 데이터를 읽기 쉽게 보여 주는 방법을 배웁니다"),
    ("C08", "파이썬 데이터 분석 기초", "데이터", 2, 4, "파이썬으로 표 데이터를 읽고 통계를 계산하며 그래프를 그리는 실습을 합니다"),
    ("C09", "통계로 읽는 부대 생활", "데이터", 2, 2, "평균과 분포, 상관관계로 생활 데이터를 해석하는 방법을 사례로 배웁니다"),
    ("C10", "데이터 품질 점검 실습", "데이터", 3, 3, "결측과 중복, 형식 오류를 찾아 고치고 데이터 신뢰도를 점검하는 방법을 실습합니다"),
    ("C11", "정보보안 기본 수칙", "보안", 1, 1, "비밀번호 관리와 외부 저장매체 사용 금지, 촬영 금지 등 보안 수칙을 배웁니다"),
    ("C12", "사이버 위협 알아보기", "보안", 1, 2, "악성 메일과 피싱 사례를 보고 의심스러운 연락에 대처하는 방법을 배웁니다"),
    ("C13", "계정과 비밀번호 관리", "보안", 2, 1, "안전한 비밀번호 만들기와 계정 점검 방법을 실습하며 보안 습관을 기릅니다"),
    ("C14", "보안 사고 대응 절차", "보안", 2, 2, "보안 사고가 의심될 때 보고하는 절차와 지휘계통 보고 요령을 사례로 배웁니다"),
    ("C15", "네트워크 보안 이해", "보안", 3, 4, "네트워크 구조와 접근 통제의 원리를 배우고 취약점 점검 개념을 이해합니다"),
    ("C16", "생활관 안전사고 예방", "안전", 1, 1, "전열기 사용과 결빙, 낙상 등 생활관에서 일어나는 안전사고 예방 수칙을 배웁니다"),
    ("C17", "응급처치 기본", "안전", 1, 2, "지혈과 심폐소생술 등 응급처치 기본 동작을 실습하고 보고 요령을 익힙니다"),
    ("C18", "화재 예방과 대피", "안전", 2, 2, "화재 원인과 초기 진화, 대피 경로 확인 방법을 훈련 사례로 배웁니다"),
    ("C19", "동절기 안전 관리", "안전", 2, 1, "한파와 결빙에 대비한 방한 수칙과 동상 예방, 안전 점검 방법을 배웁니다"),
    ("C20", "안전 리더 과정", "안전", 3, 3, "분대 단위 안전 점검표를 만들고 위험 요인을 찾아 개선하는 방법을 실습합니다"),
    ("C21", "분대장 리더십 기초", "리더십", 1, 2, "분대원을 이끄는 기본 자세와 지시, 보고 요령을 사례로 배웁니다"),
    ("C22", "팀 목표 세우기", "리더십", 1, 1, "팀의 목표를 정하고 역할을 나누며 진행 상황을 점검하는 방법을 배웁니다"),
    ("C23", "갈등 조정과 중재", "리더십", 2, 2, "생활관에서 생기는 갈등을 듣고 조정하는 대화 방법을 역할극으로 연습합니다"),
    ("C24", "멘토링과 후임 지도", "리더십", 2, 2, "후임을 지도하고 격려하는 멘토링 방법과 피드백 요령을 사례로 배웁니다"),
    ("C25", "지휘 판단과 의사결정", "리더십", 3, 3, "정보가 부족할 때 근거를 모아 결정하고 책임 있게 보고하는 방법을 사례로 배웁니다"),
    ("C26", "기초 체력 향상 프로그램", "체력", 1, 2, "팔굽혀펴기와 윗몸일으키기, 달리기 기록을 단계별로 올리는 체력 훈련을 합니다"),
    ("C27", "스트레칭과 부상 예방", "체력", 1, 1, "운동 전후 스트레칭과 올바른 자세로 근육 부상을 예방하는 방법을 배웁니다"),
    ("C28", "체력 측정 대비 훈련", "체력", 2, 2, "체력 측정 종목별 요령과 기록 향상 훈련 계획을 세우고 실천합니다"),
    ("C29", "영양과 회복 관리", "체력", 2, 1, "훈련 후 영양 섭취와 수면, 회복 관리로 체력을 유지하는 방법을 배웁니다"),
    ("C30", "전투 체력 심화 과정", "체력", 3, 3, "장거리 달리기와 근지구력을 높이는 심화 체력 훈련과 기록 관리 방법을 배웁니다"),
    ("C31", "컴퓨터활용능력 대비", "자격증 대비", 1, 4, "표 계산과 문서 작성 등 컴퓨터활용능력 시험 문제 풀이와 자격증 대비 요령을 익힙니다"),
    ("C32", "한국사능력 대비", "자격증 대비", 1, 3, "한국사 흐름을 정리하고 기출 문제를 풀며 자격증 시험을 대비합니다"),
    ("C33", "정보처리 기초 대비", "자격증 대비", 2, 4, "자료구조와 알고리즘, 데이터베이스 기초를 정리하고 자격증 시험 문제를 풉니다"),
    ("C34", "데이터 분석 자격 대비", "자격증 대비", 2, 4, "통계와 데이터 처리 개념을 정리하고 데이터 분석 자격증 시험을 대비합니다"),
    ("C35", "전기·정비 기능사 대비", "자격증 대비", 3, 4, "전기 회로와 정비 이론을 정리하고 실기 요령을 익혀 기능사 자격증을 대비합니다"),
    ("C36", "말하기와 발표 기초", "소통·정서", 1, 1, "핵심을 먼저 말하는 보고 방식과 발표 자세를 연습하며 소통 능력을 기릅니다"),
    ("C37", "스트레스 관리와 마음 건강", "소통·정서", 1, 2, "스트레스 신호를 알아차리고 상담 창구를 이용하는 방법 등 마음 건강 관리를 배웁니다"),
    ("C38", "경청과 공감 대화", "소통·정서", 2, 2, "상대의 말을 끝까지 듣고 공감하며 답하는 대화 방법을 짝 연습으로 익힙니다"),
    ("C39", "전역 후 진로 설계", "소통·정서", 2, 2, "전역 후 진로를 정하고 자기개발 계획과 자격증 준비 일정을 세우는 방법을 배웁니다"),
    ("C40", "부대 보고서 글쓰기", "소통·정서", 3, 3, "읽는 사람을 생각해 근거와 결론이 분명한 보고서와 공문을 쓰는 방법을 실습합니다"),
]
courses = pd.DataFrame(COURSE_ROWS, columns=["id", "title", "topic", "level", "hours", "desc"])
CID = list(courses["id"])
RANKS = ["이병", "일병", "상병", "병장"]

# 가상 장병 60명: 계급, 관심 주제 2개, 수강 이력과 평점(1~5)을 난수로 만든다
rng = np.random.default_rng(13)
item_pop = rng.lognormal(0, 0.6, len(courses))
rows, soldier_rows = [], []
for i in range(60):
    sid = f"S{i + 1:02d}"
    rank = RANKS[min(i // 15, 3)]
    interest = list(rng.choice(TOPICS, 2, replace=False))
    max_level = 1 + (RANKS.index(rank) >= 1) + (RANKS.index(rank) >= 3)
    w = np.array([item_pop[j] * (1 + 7 * (courses.topic[j] in interest)) * (1.0 if courses.level[j] <= max_level else 0.3)
                  for j in range(len(courses))])
    n_take = int(rng.integers(7, 14))
    picked = rng.choice(len(courses), n_take, replace=False, p=w / w.sum())
    for j in picked:
        score = 2.9 + 1.3 * (courses.topic[j] in interest) + rng.normal(0, 0.7)
        rows.append((sid, CID[j], int(np.clip(round(score), 1, 5))))
    soldier_rows.append((sid, rank, interest[0], interest[1]))
soldiers = pd.DataFrame(soldier_rows, columns=["id", "rank", "interest1", "interest2"])
ratings = pd.DataFrame(rows, columns=["soldier", "course", "rating"])
SID = list(soldiers["id"])

# 평가용 분리: 평점 4 이상인 수강 중 장병마다 최대 2개를 숨겨 두고(test), 나머지로 추천 모델을 만든다(train)
test_idx = []
for sid, g in ratings[ratings.rating >= 4].groupby("soldier"):
    test_idx += list(rng.choice(g.index, min(2, len(g)), replace=False))
test = ratings.loc[sorted(test_idx)].reset_index(drop=True)
train = ratings.drop(index=test_idx).reset_index(drop=True)
R = train.pivot(index="soldier", columns="course", values="rating").reindex(index=SID, columns=CID)
M = (~R.isna()).values.astype(float)        # 수강 여부(1/0)
RV = R.fillna(0).values                      # 평점 행렬(미수강은 0)
relevant = test.groupby("soldier")["course"].apply(set).to_dict()

stat = train.groupby("course")["rating"].agg(mean="mean", count="count")
stat = stat.reindex(CID).fillna({"mean": 0, "count": 0})
g = train["rating"].mean()
# 수강 수가 적으면 전체 평균(g) 쪽으로 당기는 보정 평균 (가상의 평점 5개를 전체 평균으로 더한 효과)
stat["score"] = (stat["count"] * stat["mean"] + 5 * g) / (stat["count"] + 5)
stat["title"] = courses.set_index("id")["title"]

print(f"전체 평균 평점 {g:.2f}")
print(f"수강 5건 미만 콘텐츠 {(stat['count'] < 5).sum()}개 - 평균 평점이 불안정하므로 보정이 필요하다")
print("\n[단순 평균 평점 상위 5]")
print(stat.sort_values(["mean", "count"], ascending=False).head(5)[["title", "mean", "count"]].round(2).to_string())
print("\n[보정 점수 상위 5]")
print(stat.sort_values("score", ascending=False).head(5)[["title", "mean", "count", "score"]].round(2).to_string())

SIDX = {s: i for i, s in enumerate(SID)}


def recommend_pop(sid, k=5):
    """보정 점수 순으로 추천하되 이미 수강한 콘텐츠는 뺀다."""
    taken = set(train.loc[train.soldier == sid, "course"])
    ranked = stat.sort_values("score", ascending=False)
    return [c for c in ranked.index if c not in taken][:k]


for sid in ["S01", "S31"]:
    me = soldiers[soldiers.id == sid].iloc[0]
    print(f"\n{sid}({me['rank']}, 관심 {me['interest1']}·{me['interest2']}) 인기도 추천:", recommend_pop(sid))

hits, n = 0, 0
for sid, rel in relevant.items():
    rc = recommend_pop(sid)
    hits += len(set(rc) & rel); n += 1
prec = hits / (5 * n)
shown = {c for sid in relevant for c in recommend_pop(sid)}
print(f"\n인기도 Baseline: Precision@5 = {prec:.3f}, 추천된 콘텐츠 {len(shown)}개/{len(CID)}개 (커버리지 {len(shown) / len(CID):.0%})")
print("모든 장병에게 거의 같은 목록이 나오므로 개인 취향이 반영되지 않는다. 이후 모델은 이 값보다 좋아야 한다.")


---

## 🎯 미션 03. 설명 유사도로 콘텐츠 추천하기
**소요시간**: 약 2.0시간

### 📋 미션 배경 및 수행 지침
#### 미션 배경 및 당위성

실제 부대 실무 현장에서는 불명확한 정보 전달이나 수동 업무 처리로 인해 불필요한 행정 소요가 발생하고 주요 공지·규정 확인이 지연되는 문제가 발생하기 쉽습니다. 따라서 **콘텐츠 설명을 TF-IDF로 벡터화해 비슷한 콘텐츠를 찾는 콘텐츠 기반 추천을 만듭니다.** 과업을 체계적으로 수행하여 현장 업무 효율성을 극대화하고 정보 누락으로 인한 혼선을 사전에 방지해야 합니다.

이 문제를 해결하기 위해 본 미션에서는 **제목·주제·설명을 글자 조각 TF-IDF로 바꾸고 콘텐츠 사이 코사인 유사도를 계산 수행하고 검증하는** 실습을 단계별로 수행합니다.

학습자는 실습가이드에 제시된 총 4개 세부 활동을 순차적으로 진행하며 핵심 산출물을 도출하고 검증 기준을 달성합니다.

**이번 미션 결과물:** `유사도 행렬, 콘텐츠 기반 추천 함수, 추천 결과와 성능` 완성

### 🔍 평가 및 검증 기준 (PASS 조건)
#### 미션 판정

**평가 구분:** 필수

**핵심 평가 기준:** [필수] 콘텐츠 기반 추천 — 설명 유사도로 추천을 구현하고 결과와 한계를 확인했는가? (PASS / FAIL)

#### 미션 PASS 조건

- 요구 산출물(유사도 행렬, 콘텐츠 기반 추천 함수, 추천 결과와 성능)의 완성도 충족


- 원문 대조 점검을 통한 사실 검증 및 정확성 확보


- 교육용 가상 데이터 활용 및 군사보안 수칙 엄수


#### 채점자 판정 메모

> 단순 결과물 생성 여부뿐만 아니라 AI 결과물의 오류를 직접 찾아내고 바로잡았는지 대조표를 확인한다.



In [ ]:
"""콘텐츠 설명을 TF-IDF로 벡터화해 비슷한 콘텐츠를 찾고, 장병이 높게 평가한 콘텐츠와 닮은 콘텐츠를 추천한다."""
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# ---------------------------------------------------------------------------
# [교육용가상자료] 가상 한빛대대 교육 콘텐츠 40개와 가상 장병 60명 (실제 부대·인원과 무관)
# ---------------------------------------------------------------------------
TOPICS = ["AI 기초", "데이터", "보안", "안전", "리더십", "체력", "자격증 대비", "소통·정서"]
LEVELS = {1: "기초", 2: "중급", 3: "심화"}
COURSE_ROWS = [
    ("C01", "인공지능 첫걸음", "AI 기초", 1, 2, "인공지능이 무엇인지 사례로 알아보고 일상과 업무에서 쓰이는 방식을 배웁니다"),
    ("C02", "생성형 AI 안전하게 쓰기", "AI 기초", 1, 1, "생성형 AI에 질문하는 방법과 가상 자료만 입력하는 보안 수칙, 답변 검증 방법을 배웁니다"),
    ("C03", "머신러닝 맛보기", "AI 기초", 2, 3, "데이터로 규칙을 학습하는 머신러닝의 원리와 분류 예측 실습을 해 봅니다"),
    ("C04", "AI 프롬프트 작성 실습", "AI 기초", 2, 2, "목적에 맞는 프롬프트를 작성하고 결과를 비교하며 개선하는 실습을 합니다"),
    ("C05", "AI 윤리와 책임", "AI 기초", 2, 1, "AI 결과의 한계와 편향, 최종 판단은 사람이 한다는 원칙을 사례로 배웁니다"),
    ("C06", "엑셀 데이터 정리 기초", "데이터", 1, 2, "표 데이터를 정리하고 합계와 평균을 구하며 오류를 찾는 방법을 배웁니다"),
    ("C07", "데이터 시각화 입문", "데이터", 1, 2, "막대그래프와 꺾은선 그래프로 데이터를 읽기 쉽게 보여 주는 방법을 배웁니다"),
    ("C08", "파이썬 데이터 분석 기초", "데이터", 2, 4, "파이썬으로 표 데이터를 읽고 통계를 계산하며 그래프를 그리는 실습을 합니다"),
    ("C09", "통계로 읽는 부대 생활", "데이터", 2, 2, "평균과 분포, 상관관계로 생활 데이터를 해석하는 방법을 사례로 배웁니다"),
    ("C10", "데이터 품질 점검 실습", "데이터", 3, 3, "결측과 중복, 형식 오류를 찾아 고치고 데이터 신뢰도를 점검하는 방법을 실습합니다"),
    ("C11", "정보보안 기본 수칙", "보안", 1, 1, "비밀번호 관리와 외부 저장매체 사용 금지, 촬영 금지 등 보안 수칙을 배웁니다"),
    ("C12", "사이버 위협 알아보기", "보안", 1, 2, "악성 메일과 피싱 사례를 보고 의심스러운 연락에 대처하는 방법을 배웁니다"),
    ("C13", "계정과 비밀번호 관리", "보안", 2, 1, "안전한 비밀번호 만들기와 계정 점검 방법을 실습하며 보안 습관을 기릅니다"),
    ("C14", "보안 사고 대응 절차", "보안", 2, 2, "보안 사고가 의심될 때 보고하는 절차와 지휘계통 보고 요령을 사례로 배웁니다"),
    ("C15", "네트워크 보안 이해", "보안", 3, 4, "네트워크 구조와 접근 통제의 원리를 배우고 취약점 점검 개념을 이해합니다"),
    ("C16", "생활관 안전사고 예방", "안전", 1, 1, "전열기 사용과 결빙, 낙상 등 생활관에서 일어나는 안전사고 예방 수칙을 배웁니다"),
    ("C17", "응급처치 기본", "안전", 1, 2, "지혈과 심폐소생술 등 응급처치 기본 동작을 실습하고 보고 요령을 익힙니다"),
    ("C18", "화재 예방과 대피", "안전", 2, 2, "화재 원인과 초기 진화, 대피 경로 확인 방법을 훈련 사례로 배웁니다"),
    ("C19", "동절기 안전 관리", "안전", 2, 1, "한파와 결빙에 대비한 방한 수칙과 동상 예방, 안전 점검 방법을 배웁니다"),
    ("C20", "안전 리더 과정", "안전", 3, 3, "분대 단위 안전 점검표를 만들고 위험 요인을 찾아 개선하는 방법을 실습합니다"),
    ("C21", "분대장 리더십 기초", "리더십", 1, 2, "분대원을 이끄는 기본 자세와 지시, 보고 요령을 사례로 배웁니다"),
    ("C22", "팀 목표 세우기", "리더십", 1, 1, "팀의 목표를 정하고 역할을 나누며 진행 상황을 점검하는 방법을 배웁니다"),
    ("C23", "갈등 조정과 중재", "리더십", 2, 2, "생활관에서 생기는 갈등을 듣고 조정하는 대화 방법을 역할극으로 연습합니다"),
    ("C24", "멘토링과 후임 지도", "리더십", 2, 2, "후임을 지도하고 격려하는 멘토링 방법과 피드백 요령을 사례로 배웁니다"),
    ("C25", "지휘 판단과 의사결정", "리더십", 3, 3, "정보가 부족할 때 근거를 모아 결정하고 책임 있게 보고하는 방법을 사례로 배웁니다"),
    ("C26", "기초 체력 향상 프로그램", "체력", 1, 2, "팔굽혀펴기와 윗몸일으키기, 달리기 기록을 단계별로 올리는 체력 훈련을 합니다"),
    ("C27", "스트레칭과 부상 예방", "체력", 1, 1, "운동 전후 스트레칭과 올바른 자세로 근육 부상을 예방하는 방법을 배웁니다"),
    ("C28", "체력 측정 대비 훈련", "체력", 2, 2, "체력 측정 종목별 요령과 기록 향상 훈련 계획을 세우고 실천합니다"),
    ("C29", "영양과 회복 관리", "체력", 2, 1, "훈련 후 영양 섭취와 수면, 회복 관리로 체력을 유지하는 방법을 배웁니다"),
    ("C30", "전투 체력 심화 과정", "체력", 3, 3, "장거리 달리기와 근지구력을 높이는 심화 체력 훈련과 기록 관리 방법을 배웁니다"),
    ("C31", "컴퓨터활용능력 대비", "자격증 대비", 1, 4, "표 계산과 문서 작성 등 컴퓨터활용능력 시험 문제 풀이와 자격증 대비 요령을 익힙니다"),
    ("C32", "한국사능력 대비", "자격증 대비", 1, 3, "한국사 흐름을 정리하고 기출 문제를 풀며 자격증 시험을 대비합니다"),
    ("C33", "정보처리 기초 대비", "자격증 대비", 2, 4, "자료구조와 알고리즘, 데이터베이스 기초를 정리하고 자격증 시험 문제를 풉니다"),
    ("C34", "데이터 분석 자격 대비", "자격증 대비", 2, 4, "통계와 데이터 처리 개념을 정리하고 데이터 분석 자격증 시험을 대비합니다"),
    ("C35", "전기·정비 기능사 대비", "자격증 대비", 3, 4, "전기 회로와 정비 이론을 정리하고 실기 요령을 익혀 기능사 자격증을 대비합니다"),
    ("C36", "말하기와 발표 기초", "소통·정서", 1, 1, "핵심을 먼저 말하는 보고 방식과 발표 자세를 연습하며 소통 능력을 기릅니다"),
    ("C37", "스트레스 관리와 마음 건강", "소통·정서", 1, 2, "스트레스 신호를 알아차리고 상담 창구를 이용하는 방법 등 마음 건강 관리를 배웁니다"),
    ("C38", "경청과 공감 대화", "소통·정서", 2, 2, "상대의 말을 끝까지 듣고 공감하며 답하는 대화 방법을 짝 연습으로 익힙니다"),
    ("C39", "전역 후 진로 설계", "소통·정서", 2, 2, "전역 후 진로를 정하고 자기개발 계획과 자격증 준비 일정을 세우는 방법을 배웁니다"),
    ("C40", "부대 보고서 글쓰기", "소통·정서", 3, 3, "읽는 사람을 생각해 근거와 결론이 분명한 보고서와 공문을 쓰는 방법을 실습합니다"),
]
courses = pd.DataFrame(COURSE_ROWS, columns=["id", "title", "topic", "level", "hours", "desc"])
CID = list(courses["id"])
RANKS = ["이병", "일병", "상병", "병장"]

# 가상 장병 60명: 계급, 관심 주제 2개, 수강 이력과 평점(1~5)을 난수로 만든다
rng = np.random.default_rng(13)
item_pop = rng.lognormal(0, 0.6, len(courses))
rows, soldier_rows = [], []
for i in range(60):
    sid = f"S{i + 1:02d}"
    rank = RANKS[min(i // 15, 3)]
    interest = list(rng.choice(TOPICS, 2, replace=False))
    max_level = 1 + (RANKS.index(rank) >= 1) + (RANKS.index(rank) >= 3)
    w = np.array([item_pop[j] * (1 + 7 * (courses.topic[j] in interest)) * (1.0 if courses.level[j] <= max_level else 0.3)
                  for j in range(len(courses))])
    n_take = int(rng.integers(7, 14))
    picked = rng.choice(len(courses), n_take, replace=False, p=w / w.sum())
    for j in picked:
        score = 2.9 + 1.3 * (courses.topic[j] in interest) + rng.normal(0, 0.7)
        rows.append((sid, CID[j], int(np.clip(round(score), 1, 5))))
    soldier_rows.append((sid, rank, interest[0], interest[1]))
soldiers = pd.DataFrame(soldier_rows, columns=["id", "rank", "interest1", "interest2"])
ratings = pd.DataFrame(rows, columns=["soldier", "course", "rating"])
SID = list(soldiers["id"])

# 평가용 분리: 평점 4 이상인 수강 중 장병마다 최대 2개를 숨겨 두고(test), 나머지로 추천 모델을 만든다(train)
test_idx = []
for sid, g in ratings[ratings.rating >= 4].groupby("soldier"):
    test_idx += list(rng.choice(g.index, min(2, len(g)), replace=False))
test = ratings.loc[sorted(test_idx)].reset_index(drop=True)
train = ratings.drop(index=test_idx).reset_index(drop=True)
R = train.pivot(index="soldier", columns="course", values="rating").reindex(index=SID, columns=CID)
M = (~R.isna()).values.astype(float)        # 수강 여부(1/0)
RV = R.fillna(0).values                      # 평점 행렬(미수강은 0)
relevant = test.groupby("soldier")["course"].apply(set).to_dict()

SIDX = {s: i for i, s in enumerate(SID)}
text = courses["title"] + " " + courses["topic"] + " " + courses["desc"]
vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 3))   # 글자 2~3개 조각 기준 TF-IDF
X = vec.fit_transform(text)
sim = cosine_similarity(X)
print(f"TF-IDF 행렬 {X.shape} (콘텐츠 x 글자 조각)")

for cid in ["C03", "C26"]:
    j = CID.index(cid)
    order = [k for k in np.argsort(-sim[j]) if k != j][:3]
    print(f"\n'{courses.title[j]}'와 비슷한 콘텐츠:")
    for k in order:
        print(f"  {courses.title[k]} ({courses.topic[k]}) 유사도 {sim[j, k]:.2f}")


def content_scores(sid):
    """평점 4 이상으로 들은 콘텐츠를 (평점-3)만큼 가중해, 그 콘텐츠들과 설명이 비슷한 정도를 점수로 한다."""
    r = RV[SIDX[sid]]
    w = np.where(r >= 4, r - 3, 0)
    if w.sum() == 0:
        return np.zeros(len(CID))
    return (sim * w).sum(axis=1) / w.sum()


def recommend(sid, k=5):
    s = np.where(M[SIDX[sid]] == 1, -1e9, content_scores(sid))
    return [CID[j] for j in np.argsort(-s)[:k]]


for sid in ["S01", "S31"]:
    me = soldiers[soldiers.id == sid].iloc[0]
    print(f"\n{sid}(관심 {me['interest1']}·{me['interest2']}) 콘텐츠 기반 추천:")
    for c in recommend(sid):
        j = CID.index(c)
        print(f"  {c} {courses.title[j]} [{courses.topic[j]}]")

hits, shown = 0, set()
for sid, rel in relevant.items():
    rc = recommend(sid)
    hits += len(set(rc) & rel); shown |= set(rc)
print(f"\n콘텐츠 기반: Precision@5 = {hits / (5 * len(relevant)):.3f}, 커버리지 {len(shown) / len(CID):.0%}")
print("이력이 있어야 추천할 수 있고, 이미 좋아한 주제와 비슷한 것 위주로 추천되는 경향이 있다.")


---

## 🎯 미션 04. 유사 장병 기반 추천 만들기
**소요시간**: 약 2.0시간

### 📋 미션 배경 및 수행 지침
#### 미션 배경 및 당위성

실제 부대 실무 현장에서는 불명확한 정보 전달이나 수동 업무 처리로 인해 불필요한 행정 소요가 발생하고 주요 공지·규정 확인이 지연되는 문제가 발생하기 쉽습니다. 따라서 **평점 패턴이 비슷한 장병의 선택을 활용하는 협업 필터링과 행렬 분해 추천을 구현합니다.** 과업을 체계적으로 수행하여 현장 업무 효율성을 극대화하고 정보 누락으로 인한 혼선을 사전에 방지해야 합니다.

이 문제를 해결하기 위해 본 미션에서는 **장병 사이 평점 유사도로 이웃 장병을 찾아 점수를 계산하는 방법과, TruncatedSVD로 평점 행렬을 숨은 취향 축으로 줄였다가 복원해 빈 칸을 채우는 방법을 제작하고 안정성을 확보하는** 실습을 단계별로 수행합니다.

학습자는 실습가이드에 제시된 총 4개 세부 활동을 순차적으로 진행하며 핵심 산출물을 도출하고 검증 기준을 달성합니다.

**이번 미션 결과물:** `협업 필터링 함수, 행렬 분해 함수, 설정값별 성능표` 완성

### 🔍 평가 및 검증 기준 (PASS 조건)
#### 미션 판정

**평가 구분:** 필수

**핵심 평가 기준:** [선택] 협업 필터링 — 유사 장병 기반 추천과 행렬 분해를 구현하고 설정값에 따른 변화를 확인했는가? (PASS / FAIL)

#### 미션 PASS 조건

- 요구 산출물(협업 필터링 함수, 행렬 분해 함수, 설정값별 성능표)의 완성도 충족


- 원문 대조 점검을 통한 사실 검증 및 정확성 확보


- 교육용 가상 데이터 활용 및 군사보안 수칙 엄수


#### 채점자 판정 메모

> 단순 결과물 생성 여부뿐만 아니라 AI 결과물의 오류를 직접 찾아내고 바로잡았는지 대조표를 확인한다.



In [ ]:
"""평점 패턴이 비슷한 장병(유사 장병)의 선택으로 추천하는 협업 필터링과, 행렬 분해(TruncatedSVD)로 추천하는 방법을 구현하고 비교한다."""
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import TruncatedSVD

# ---------------------------------------------------------------------------
# [교육용가상자료] 가상 한빛대대 교육 콘텐츠 40개와 가상 장병 60명 (실제 부대·인원과 무관)
# ---------------------------------------------------------------------------
TOPICS = ["AI 기초", "데이터", "보안", "안전", "리더십", "체력", "자격증 대비", "소통·정서"]
LEVELS = {1: "기초", 2: "중급", 3: "심화"}
COURSE_ROWS = [
    ("C01", "인공지능 첫걸음", "AI 기초", 1, 2, "인공지능이 무엇인지 사례로 알아보고 일상과 업무에서 쓰이는 방식을 배웁니다"),
    ("C02", "생성형 AI 안전하게 쓰기", "AI 기초", 1, 1, "생성형 AI에 질문하는 방법과 가상 자료만 입력하는 보안 수칙, 답변 검증 방법을 배웁니다"),
    ("C03", "머신러닝 맛보기", "AI 기초", 2, 3, "데이터로 규칙을 학습하는 머신러닝의 원리와 분류 예측 실습을 해 봅니다"),
    ("C04", "AI 프롬프트 작성 실습", "AI 기초", 2, 2, "목적에 맞는 프롬프트를 작성하고 결과를 비교하며 개선하는 실습을 합니다"),
    ("C05", "AI 윤리와 책임", "AI 기초", 2, 1, "AI 결과의 한계와 편향, 최종 판단은 사람이 한다는 원칙을 사례로 배웁니다"),
    ("C06", "엑셀 데이터 정리 기초", "데이터", 1, 2, "표 데이터를 정리하고 합계와 평균을 구하며 오류를 찾는 방법을 배웁니다"),
    ("C07", "데이터 시각화 입문", "데이터", 1, 2, "막대그래프와 꺾은선 그래프로 데이터를 읽기 쉽게 보여 주는 방법을 배웁니다"),
    ("C08", "파이썬 데이터 분석 기초", "데이터", 2, 4, "파이썬으로 표 데이터를 읽고 통계를 계산하며 그래프를 그리는 실습을 합니다"),
    ("C09", "통계로 읽는 부대 생활", "데이터", 2, 2, "평균과 분포, 상관관계로 생활 데이터를 해석하는 방법을 사례로 배웁니다"),
    ("C10", "데이터 품질 점검 실습", "데이터", 3, 3, "결측과 중복, 형식 오류를 찾아 고치고 데이터 신뢰도를 점검하는 방법을 실습합니다"),
    ("C11", "정보보안 기본 수칙", "보안", 1, 1, "비밀번호 관리와 외부 저장매체 사용 금지, 촬영 금지 등 보안 수칙을 배웁니다"),
    ("C12", "사이버 위협 알아보기", "보안", 1, 2, "악성 메일과 피싱 사례를 보고 의심스러운 연락에 대처하는 방법을 배웁니다"),
    ("C13", "계정과 비밀번호 관리", "보안", 2, 1, "안전한 비밀번호 만들기와 계정 점검 방법을 실습하며 보안 습관을 기릅니다"),
    ("C14", "보안 사고 대응 절차", "보안", 2, 2, "보안 사고가 의심될 때 보고하는 절차와 지휘계통 보고 요령을 사례로 배웁니다"),
    ("C15", "네트워크 보안 이해", "보안", 3, 4, "네트워크 구조와 접근 통제의 원리를 배우고 취약점 점검 개념을 이해합니다"),
    ("C16", "생활관 안전사고 예방", "안전", 1, 1, "전열기 사용과 결빙, 낙상 등 생활관에서 일어나는 안전사고 예방 수칙을 배웁니다"),
    ("C17", "응급처치 기본", "안전", 1, 2, "지혈과 심폐소생술 등 응급처치 기본 동작을 실습하고 보고 요령을 익힙니다"),
    ("C18", "화재 예방과 대피", "안전", 2, 2, "화재 원인과 초기 진화, 대피 경로 확인 방법을 훈련 사례로 배웁니다"),
    ("C19", "동절기 안전 관리", "안전", 2, 1, "한파와 결빙에 대비한 방한 수칙과 동상 예방, 안전 점검 방법을 배웁니다"),
    ("C20", "안전 리더 과정", "안전", 3, 3, "분대 단위 안전 점검표를 만들고 위험 요인을 찾아 개선하는 방법을 실습합니다"),
    ("C21", "분대장 리더십 기초", "리더십", 1, 2, "분대원을 이끄는 기본 자세와 지시, 보고 요령을 사례로 배웁니다"),
    ("C22", "팀 목표 세우기", "리더십", 1, 1, "팀의 목표를 정하고 역할을 나누며 진행 상황을 점검하는 방법을 배웁니다"),
    ("C23", "갈등 조정과 중재", "리더십", 2, 2, "생활관에서 생기는 갈등을 듣고 조정하는 대화 방법을 역할극으로 연습합니다"),
    ("C24", "멘토링과 후임 지도", "리더십", 2, 2, "후임을 지도하고 격려하는 멘토링 방법과 피드백 요령을 사례로 배웁니다"),
    ("C25", "지휘 판단과 의사결정", "리더십", 3, 3, "정보가 부족할 때 근거를 모아 결정하고 책임 있게 보고하는 방법을 사례로 배웁니다"),
    ("C26", "기초 체력 향상 프로그램", "체력", 1, 2, "팔굽혀펴기와 윗몸일으키기, 달리기 기록을 단계별로 올리는 체력 훈련을 합니다"),
    ("C27", "스트레칭과 부상 예방", "체력", 1, 1, "운동 전후 스트레칭과 올바른 자세로 근육 부상을 예방하는 방법을 배웁니다"),
    ("C28", "체력 측정 대비 훈련", "체력", 2, 2, "체력 측정 종목별 요령과 기록 향상 훈련 계획을 세우고 실천합니다"),
    ("C29", "영양과 회복 관리", "체력", 2, 1, "훈련 후 영양 섭취와 수면, 회복 관리로 체력을 유지하는 방법을 배웁니다"),
    ("C30", "전투 체력 심화 과정", "체력", 3, 3, "장거리 달리기와 근지구력을 높이는 심화 체력 훈련과 기록 관리 방법을 배웁니다"),
    ("C31", "컴퓨터활용능력 대비", "자격증 대비", 1, 4, "표 계산과 문서 작성 등 컴퓨터활용능력 시험 문제 풀이와 자격증 대비 요령을 익힙니다"),
    ("C32", "한국사능력 대비", "자격증 대비", 1, 3, "한국사 흐름을 정리하고 기출 문제를 풀며 자격증 시험을 대비합니다"),
    ("C33", "정보처리 기초 대비", "자격증 대비", 2, 4, "자료구조와 알고리즘, 데이터베이스 기초를 정리하고 자격증 시험 문제를 풉니다"),
    ("C34", "데이터 분석 자격 대비", "자격증 대비", 2, 4, "통계와 데이터 처리 개념을 정리하고 데이터 분석 자격증 시험을 대비합니다"),
    ("C35", "전기·정비 기능사 대비", "자격증 대비", 3, 4, "전기 회로와 정비 이론을 정리하고 실기 요령을 익혀 기능사 자격증을 대비합니다"),
    ("C36", "말하기와 발표 기초", "소통·정서", 1, 1, "핵심을 먼저 말하는 보고 방식과 발표 자세를 연습하며 소통 능력을 기릅니다"),
    ("C37", "스트레스 관리와 마음 건강", "소통·정서", 1, 2, "스트레스 신호를 알아차리고 상담 창구를 이용하는 방법 등 마음 건강 관리를 배웁니다"),
    ("C38", "경청과 공감 대화", "소통·정서", 2, 2, "상대의 말을 끝까지 듣고 공감하며 답하는 대화 방법을 짝 연습으로 익힙니다"),
    ("C39", "전역 후 진로 설계", "소통·정서", 2, 2, "전역 후 진로를 정하고 자기개발 계획과 자격증 준비 일정을 세우는 방법을 배웁니다"),
    ("C40", "부대 보고서 글쓰기", "소통·정서", 3, 3, "읽는 사람을 생각해 근거와 결론이 분명한 보고서와 공문을 쓰는 방법을 실습합니다"),
]
courses = pd.DataFrame(COURSE_ROWS, columns=["id", "title", "topic", "level", "hours", "desc"])
CID = list(courses["id"])
RANKS = ["이병", "일병", "상병", "병장"]

# 가상 장병 60명: 계급, 관심 주제 2개, 수강 이력과 평점(1~5)을 난수로 만든다
rng = np.random.default_rng(13)
item_pop = rng.lognormal(0, 0.6, len(courses))
rows, soldier_rows = [], []
for i in range(60):
    sid = f"S{i + 1:02d}"
    rank = RANKS[min(i // 15, 3)]
    interest = list(rng.choice(TOPICS, 2, replace=False))
    max_level = 1 + (RANKS.index(rank) >= 1) + (RANKS.index(rank) >= 3)
    w = np.array([item_pop[j] * (1 + 7 * (courses.topic[j] in interest)) * (1.0 if courses.level[j] <= max_level else 0.3)
                  for j in range(len(courses))])
    n_take = int(rng.integers(7, 14))
    picked = rng.choice(len(courses), n_take, replace=False, p=w / w.sum())
    for j in picked:
        score = 2.9 + 1.3 * (courses.topic[j] in interest) + rng.normal(0, 0.7)
        rows.append((sid, CID[j], int(np.clip(round(score), 1, 5))))
    soldier_rows.append((sid, rank, interest[0], interest[1]))
soldiers = pd.DataFrame(soldier_rows, columns=["id", "rank", "interest1", "interest2"])
ratings = pd.DataFrame(rows, columns=["soldier", "course", "rating"])
SID = list(soldiers["id"])

# 평가용 분리: 평점 4 이상인 수강 중 장병마다 최대 2개를 숨겨 두고(test), 나머지로 추천 모델을 만든다(train)
test_idx = []
for sid, g in ratings[ratings.rating >= 4].groupby("soldier"):
    test_idx += list(rng.choice(g.index, min(2, len(g)), replace=False))
test = ratings.loc[sorted(test_idx)].reset_index(drop=True)
train = ratings.drop(index=test_idx).reset_index(drop=True)
R = train.pivot(index="soldier", columns="course", values="rating").reindex(index=SID, columns=CID)
M = (~R.isna()).values.astype(float)        # 수강 여부(1/0)
RV = R.fillna(0).values                      # 평점 행렬(미수강은 0)
relevant = test.groupby("soldier")["course"].apply(set).to_dict()

SIDX = {s: i for i, s in enumerate(SID)}
user_sim = cosine_similarity(RV)       # 장병 사이의 평점 패턴 유사도
np.fill_diagonal(user_sim, 0)          # 자기 자신은 제외


def cf_scores(sid, n=10):
    """유사도가 높은 장병 n명의 평점을 유사도로 가중 평균한다."""
    i = SIDX[sid]
    nb = np.argsort(-user_sim[i])[:n]
    return (user_sim[i, nb][:, None] * RV[nb]).sum(axis=0) / user_sim[i, nb].sum()


def svd_scores_model(k):
    """평점 행렬을 k개의 숨은 취향 축으로 줄였다가 복원해 비어 있던 칸의 점수를 채운다."""
    svd = TruncatedSVD(n_components=k, random_state=13)
    U = svd.fit_transform(RV)
    return U @ svd.components_


def recommend(scores, sid, k=5):
    s = np.where(M[SIDX[sid]] == 1, -1e9, scores)
    return [CID[j] for j in np.argsort(-s)[:k]]


def prec_at5(score_fn):
    hits, shown = 0, set()
    for sid, rel in relevant.items():
        rc = recommend(score_fn(sid), sid)
        hits += len(set(rc) & rel); shown |= set(rc)
    return hits / (5 * len(relevant)), len(shown) / len(CID)


i = SIDX["S01"]
nb = np.argsort(-user_sim[i])[:3]
print("S01과 비슷한 장병:", [(SID[j], round(float(user_sim[i, j]), 2), str(soldiers.interest1[j])) for j in nb])
print("S01 유사 장병 기반 추천:", [courses.title[CID.index(c)] for c in recommend(cf_scores("S01"), "S01")])

print("\n[이웃 수(n)별 Precision@5, 커버리지]")
for n in [3, 5, 10, 20, 40]:
    p, cv = prec_at5(lambda s: cf_scores(s, n))
    print(f"  n={n:>2}: {p:.3f}, {cv:.0%}")

print("\n[숨은 취향 축 수(k)별 행렬 분해 Precision@5, 커버리지]")
for k in [2, 4, 6, 10]:
    S = svd_scores_model(k)
    p, cv = prec_at5(lambda s: S[SIDX[s]])
    print(f"  k={k:>2}: {p:.3f}, {cv:.0%}")

svd = TruncatedSVD(n_components=6, random_state=13).fit(RV)
print(f"\n축 6개가 설명하는 변동 비율: {svd.explained_variance_ratio_.sum():.0%}")
print("유사 장병이 적은 콘텐츠(수강 수 1~2건)는 추천되기 어렵고, 이력이 없는 신규 장병은 유사 장병을 찾을 수 없다.")


---

## 🎯 미션 05. 추천 성능 평가하기
**소요시간**: 약 2.0시간

### 📋 미션 배경 및 수행 지침
#### 미션 배경 및 당위성

실제 부대 실무 현장에서는 불명확한 정보 전달이나 수동 업무 처리로 인해 불필요한 행정 소요가 발생하고 주요 공지·규정 확인이 지연되는 문제가 발생하기 쉽습니다. 따라서 **네 가지 추천 모델을 같은 기준에서 Precision@5, Recall@5, 커버리지로 평가합니다.** 과업을 체계적으로 수행하여 현장 업무 효율성을 극대화하고 정보 누락으로 인한 혼선을 사전에 방지해야 합니다.

이 문제를 해결하기 위해 본 미션에서는 **장병마다 숨겨 둔 평점 4 이상 수강을 얼마나 맞히는지 모델별로 계산하고, 추천 개수(k)와 계급별 성능을 비교 수행하고 검증하는** 실습을 단계별로 수행합니다.

학습자는 실습가이드에 제시된 총 4개 세부 활동을 순차적으로 진행하며 핵심 산출물을 도출하고 검증 기준을 달성합니다.

**이번 미션 결과물:** `모델 비교표, k별·계급별 성능, 비교 그래프, 목표 판정` 완성

### 🔍 평가 및 검증 기준 (PASS 조건)
#### 미션 판정

**평가 구분:** 필수

**핵심 평가 기준:** [필수] 성능 평가 — 평가 지표를 계산해 모델을 비교하고 목표 달성 여부를 근거와 함께 판정했는가? (PASS / FAIL)

#### 미션 PASS 조건

- 요구 산출물(모델 비교표, k별·계급별 성능, 비교 그래프, 목표 판정)의 완성도 충족


- 원문 대조 점검을 통한 사실 검증 및 정확성 확보


- 교육용 가상 데이터 활용 및 군사보안 수칙 엄수


#### 채점자 판정 메모

> 단순 결과물 생성 여부뿐만 아니라 AI 결과물의 오류를 직접 찾아내고 바로잡았는지 대조표를 확인한다.



In [ ]:
"""인기도·콘텐츠 기반·유사 장병·행렬 분해 네 가지 추천을 Precision@5, Recall@5, 커버리지로 같은 기준에서 비교하고 목표 달성 여부를 판정한다."""
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import TruncatedSVD

# ---------------------------------------------------------------------------
# [교육용가상자료] 가상 한빛대대 교육 콘텐츠 40개와 가상 장병 60명 (실제 부대·인원과 무관)
# ---------------------------------------------------------------------------
TOPICS = ["AI 기초", "데이터", "보안", "안전", "리더십", "체력", "자격증 대비", "소통·정서"]
LEVELS = {1: "기초", 2: "중급", 3: "심화"}
COURSE_ROWS = [
    ("C01", "인공지능 첫걸음", "AI 기초", 1, 2, "인공지능이 무엇인지 사례로 알아보고 일상과 업무에서 쓰이는 방식을 배웁니다"),
    ("C02", "생성형 AI 안전하게 쓰기", "AI 기초", 1, 1, "생성형 AI에 질문하는 방법과 가상 자료만 입력하는 보안 수칙, 답변 검증 방법을 배웁니다"),
    ("C03", "머신러닝 맛보기", "AI 기초", 2, 3, "데이터로 규칙을 학습하는 머신러닝의 원리와 분류 예측 실습을 해 봅니다"),
    ("C04", "AI 프롬프트 작성 실습", "AI 기초", 2, 2, "목적에 맞는 프롬프트를 작성하고 결과를 비교하며 개선하는 실습을 합니다"),
    ("C05", "AI 윤리와 책임", "AI 기초", 2, 1, "AI 결과의 한계와 편향, 최종 판단은 사람이 한다는 원칙을 사례로 배웁니다"),
    ("C06", "엑셀 데이터 정리 기초", "데이터", 1, 2, "표 데이터를 정리하고 합계와 평균을 구하며 오류를 찾는 방법을 배웁니다"),
    ("C07", "데이터 시각화 입문", "데이터", 1, 2, "막대그래프와 꺾은선 그래프로 데이터를 읽기 쉽게 보여 주는 방법을 배웁니다"),
    ("C08", "파이썬 데이터 분석 기초", "데이터", 2, 4, "파이썬으로 표 데이터를 읽고 통계를 계산하며 그래프를 그리는 실습을 합니다"),
    ("C09", "통계로 읽는 부대 생활", "데이터", 2, 2, "평균과 분포, 상관관계로 생활 데이터를 해석하는 방법을 사례로 배웁니다"),
    ("C10", "데이터 품질 점검 실습", "데이터", 3, 3, "결측과 중복, 형식 오류를 찾아 고치고 데이터 신뢰도를 점검하는 방법을 실습합니다"),
    ("C11", "정보보안 기본 수칙", "보안", 1, 1, "비밀번호 관리와 외부 저장매체 사용 금지, 촬영 금지 등 보안 수칙을 배웁니다"),
    ("C12", "사이버 위협 알아보기", "보안", 1, 2, "악성 메일과 피싱 사례를 보고 의심스러운 연락에 대처하는 방법을 배웁니다"),
    ("C13", "계정과 비밀번호 관리", "보안", 2, 1, "안전한 비밀번호 만들기와 계정 점검 방법을 실습하며 보안 습관을 기릅니다"),
    ("C14", "보안 사고 대응 절차", "보안", 2, 2, "보안 사고가 의심될 때 보고하는 절차와 지휘계통 보고 요령을 사례로 배웁니다"),
    ("C15", "네트워크 보안 이해", "보안", 3, 4, "네트워크 구조와 접근 통제의 원리를 배우고 취약점 점검 개념을 이해합니다"),
    ("C16", "생활관 안전사고 예방", "안전", 1, 1, "전열기 사용과 결빙, 낙상 등 생활관에서 일어나는 안전사고 예방 수칙을 배웁니다"),
    ("C17", "응급처치 기본", "안전", 1, 2, "지혈과 심폐소생술 등 응급처치 기본 동작을 실습하고 보고 요령을 익힙니다"),
    ("C18", "화재 예방과 대피", "안전", 2, 2, "화재 원인과 초기 진화, 대피 경로 확인 방법을 훈련 사례로 배웁니다"),
    ("C19", "동절기 안전 관리", "안전", 2, 1, "한파와 결빙에 대비한 방한 수칙과 동상 예방, 안전 점검 방법을 배웁니다"),
    ("C20", "안전 리더 과정", "안전", 3, 3, "분대 단위 안전 점검표를 만들고 위험 요인을 찾아 개선하는 방법을 실습합니다"),
    ("C21", "분대장 리더십 기초", "리더십", 1, 2, "분대원을 이끄는 기본 자세와 지시, 보고 요령을 사례로 배웁니다"),
    ("C22", "팀 목표 세우기", "리더십", 1, 1, "팀의 목표를 정하고 역할을 나누며 진행 상황을 점검하는 방법을 배웁니다"),
    ("C23", "갈등 조정과 중재", "리더십", 2, 2, "생활관에서 생기는 갈등을 듣고 조정하는 대화 방법을 역할극으로 연습합니다"),
    ("C24", "멘토링과 후임 지도", "리더십", 2, 2, "후임을 지도하고 격려하는 멘토링 방법과 피드백 요령을 사례로 배웁니다"),
    ("C25", "지휘 판단과 의사결정", "리더십", 3, 3, "정보가 부족할 때 근거를 모아 결정하고 책임 있게 보고하는 방법을 사례로 배웁니다"),
    ("C26", "기초 체력 향상 프로그램", "체력", 1, 2, "팔굽혀펴기와 윗몸일으키기, 달리기 기록을 단계별로 올리는 체력 훈련을 합니다"),
    ("C27", "스트레칭과 부상 예방", "체력", 1, 1, "운동 전후 스트레칭과 올바른 자세로 근육 부상을 예방하는 방법을 배웁니다"),
    ("C28", "체력 측정 대비 훈련", "체력", 2, 2, "체력 측정 종목별 요령과 기록 향상 훈련 계획을 세우고 실천합니다"),
    ("C29", "영양과 회복 관리", "체력", 2, 1, "훈련 후 영양 섭취와 수면, 회복 관리로 체력을 유지하는 방법을 배웁니다"),
    ("C30", "전투 체력 심화 과정", "체력", 3, 3, "장거리 달리기와 근지구력을 높이는 심화 체력 훈련과 기록 관리 방법을 배웁니다"),
    ("C31", "컴퓨터활용능력 대비", "자격증 대비", 1, 4, "표 계산과 문서 작성 등 컴퓨터활용능력 시험 문제 풀이와 자격증 대비 요령을 익힙니다"),
    ("C32", "한국사능력 대비", "자격증 대비", 1, 3, "한국사 흐름을 정리하고 기출 문제를 풀며 자격증 시험을 대비합니다"),
    ("C33", "정보처리 기초 대비", "자격증 대비", 2, 4, "자료구조와 알고리즘, 데이터베이스 기초를 정리하고 자격증 시험 문제를 풉니다"),
    ("C34", "데이터 분석 자격 대비", "자격증 대비", 2, 4, "통계와 데이터 처리 개념을 정리하고 데이터 분석 자격증 시험을 대비합니다"),
    ("C35", "전기·정비 기능사 대비", "자격증 대비", 3, 4, "전기 회로와 정비 이론을 정리하고 실기 요령을 익혀 기능사 자격증을 대비합니다"),
    ("C36", "말하기와 발표 기초", "소통·정서", 1, 1, "핵심을 먼저 말하는 보고 방식과 발표 자세를 연습하며 소통 능력을 기릅니다"),
    ("C37", "스트레스 관리와 마음 건강", "소통·정서", 1, 2, "스트레스 신호를 알아차리고 상담 창구를 이용하는 방법 등 마음 건강 관리를 배웁니다"),
    ("C38", "경청과 공감 대화", "소통·정서", 2, 2, "상대의 말을 끝까지 듣고 공감하며 답하는 대화 방법을 짝 연습으로 익힙니다"),
    ("C39", "전역 후 진로 설계", "소통·정서", 2, 2, "전역 후 진로를 정하고 자기개발 계획과 자격증 준비 일정을 세우는 방법을 배웁니다"),
    ("C40", "부대 보고서 글쓰기", "소통·정서", 3, 3, "읽는 사람을 생각해 근거와 결론이 분명한 보고서와 공문을 쓰는 방법을 실습합니다"),
]
courses = pd.DataFrame(COURSE_ROWS, columns=["id", "title", "topic", "level", "hours", "desc"])
CID = list(courses["id"])
RANKS = ["이병", "일병", "상병", "병장"]

# 가상 장병 60명: 계급, 관심 주제 2개, 수강 이력과 평점(1~5)을 난수로 만든다
rng = np.random.default_rng(13)
item_pop = rng.lognormal(0, 0.6, len(courses))
rows, soldier_rows = [], []
for i in range(60):
    sid = f"S{i + 1:02d}"
    rank = RANKS[min(i // 15, 3)]
    interest = list(rng.choice(TOPICS, 2, replace=False))
    max_level = 1 + (RANKS.index(rank) >= 1) + (RANKS.index(rank) >= 3)
    w = np.array([item_pop[j] * (1 + 7 * (courses.topic[j] in interest)) * (1.0 if courses.level[j] <= max_level else 0.3)
                  for j in range(len(courses))])
    n_take = int(rng.integers(7, 14))
    picked = rng.choice(len(courses), n_take, replace=False, p=w / w.sum())
    for j in picked:
        score = 2.9 + 1.3 * (courses.topic[j] in interest) + rng.normal(0, 0.7)
        rows.append((sid, CID[j], int(np.clip(round(score), 1, 5))))
    soldier_rows.append((sid, rank, interest[0], interest[1]))
soldiers = pd.DataFrame(soldier_rows, columns=["id", "rank", "interest1", "interest2"])
ratings = pd.DataFrame(rows, columns=["soldier", "course", "rating"])
SID = list(soldiers["id"])

# 평가용 분리: 평점 4 이상인 수강 중 장병마다 최대 2개를 숨겨 두고(test), 나머지로 추천 모델을 만든다(train)
test_idx = []
for sid, g in ratings[ratings.rating >= 4].groupby("soldier"):
    test_idx += list(rng.choice(g.index, min(2, len(g)), replace=False))
test = ratings.loc[sorted(test_idx)].reset_index(drop=True)
train = ratings.drop(index=test_idx).reset_index(drop=True)
R = train.pivot(index="soldier", columns="course", values="rating").reindex(index=SID, columns=CID)
M = (~R.isna()).values.astype(float)        # 수강 여부(1/0)
RV = R.fillna(0).values                      # 평점 행렬(미수강은 0)
relevant = test.groupby("soldier")["course"].apply(set).to_dict()

# ---- 공통 함수: 점수 계산(인기도·콘텐츠·협업·행렬 분해), 상위 추천, 평가 ----
SIDX = {s: i for i, s in enumerate(SID)}
GLOBAL_MEAN = train["rating"].mean()
_stat = train.groupby("course")["rating"].agg(["mean", "count"]).reindex(CID).fillna({"mean": GLOBAL_MEAN, "count": 0})
POP_SCORE = ((_stat["count"] * _stat["mean"] + 5 * GLOBAL_MEAN) / (_stat["count"] + 5)).values  # 수강 수가 적으면 전체 평균 쪽으로 보정

_vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 3))
TFIDF = _vec.fit_transform(courses["title"] + " " + courses["topic"] + " " + courses["desc"])
ITEM_SIM = cosine_similarity(TFIDF)
USER_SIM = cosine_similarity(RV)
np.fill_diagonal(USER_SIM, 0)
_svd = TruncatedSVD(n_components=6, random_state=13)
_U = _svd.fit_transform(RV)
SVD_SCORE = _U @ _svd.components_


def pop_scores(sid):
    return POP_SCORE.copy()


def content_scores(sid):
    """평점 4 이상으로 수강한 콘텐츠의 설명과 비슷한 콘텐츠일수록 높은 점수"""
    r = RV[SIDX[sid]]
    w = np.where(r >= 4, r - 3, 0)
    if w.sum() == 0:
        return np.zeros(len(CID))
    return (ITEM_SIM * w).sum(axis=1) / w.sum()


def cf_scores(sid, n=10):
    """나와 평점 패턴이 비슷한 장병 n명이 높게 준 콘텐츠일수록 높은 점수"""
    i = SIDX[sid]
    nb = np.argsort(-USER_SIM[i])[:n]
    return (USER_SIM[i, nb][:, None] * RV[nb]).sum(axis=0) / USER_SIM[i, nb].sum()


def svd_scores(sid):
    return SVD_SCORE[SIDX[sid]].copy()


def norm(v, mask):
    """아직 안 들은 콘텐츠들 사이에서 0~1로 맞춘다"""
    x = v.copy()
    lo, hi = x[mask].min(), x[mask].max()
    return np.where(mask, (x - lo) / (hi - lo + 1e-9), 0)


def hybrid_scores(sid, w=(0.2, 0.4, 0.4)):
    unseen = M[SIDX[sid]] == 0
    return (w[0] * norm(pop_scores(sid), unseen) + w[1] * norm(content_scores(sid), unseen)
            + w[2] * norm(cf_scores(sid), unseen))


def top_k(scores, sid, k=5):
    s = np.where(M[SIDX[sid]] == 1, -1e9, scores)    # 이미 들은 콘텐츠는 제외
    return [CID[j] for j in np.argsort(-s)[:k]]


def evaluate(score_fn, k=5, sids=None):
    """장병마다 상위 k개를 추천해 정밀도(Precision@k), 재현율(Recall@k), 커버리지를 구한다."""
    prec, rec, shown = [], [], set()
    for sid, rel in relevant.items():
        if sids is not None and sid not in sids:
            continue
        rc = top_k(score_fn(sid), sid, k)
        hit = len(set(rc) & rel)
        prec.append(hit / k); rec.append(hit / len(rel)); shown |= set(rc)
    return np.mean(prec), np.mean(rec), len(shown) / len(CID)

models = {"인기도": pop_scores, "콘텐츠 기반": content_scores, "유사 장병": cf_scores, "행렬 분해": svd_scores}
rows = []
for name, fn in models.items():
    p, r, cv = evaluate(fn, 5)
    rows.append((name, p, r, cv))
res = pd.DataFrame(rows, columns=["모델", "Precision@5", "Recall@5", "커버리지"]).set_index("모델")
print("[모델 비교] 장병", len(relevant), "명, 장병마다 숨겨 둔 평점 4 이상 수강 1~2개를 맞히는지 평가")
print(res.round(3).to_string())

print("\n[추천 개수 k별 Precision / Recall - 콘텐츠 기반]")
for k in [3, 5, 10]:
    p, r, cv = evaluate(content_scores, k)
    print(f"  k={k:>2}: Precision {p:.3f}, Recall {r:.3f}")

print("\n[계급별 Precision@5 - 콘텐츠 기반]")
for rk in RANKS:
    ids = set(soldiers.loc[soldiers["rank"] == rk, "id"])
    print(f"  {rk}: {evaluate(content_scores, 5, ids)[0]:.3f}")

base = res.loc["인기도", "Precision@5"]
best = res["Precision@5"].idxmax()
print(f"\n최고 모델: {best} (Precision@5 {res.loc[best, 'Precision@5']:.3f}, 인기도의 {res.loc[best, 'Precision@5'] / base:.1f}배)")
goal = res.loc[best, "Precision@5"] >= 2 * base
print("목표(인기도 Baseline의 2배 이상):", "목표 달성" if goal else "목표 미달")
print("Recall이 낮아 보여도 정답이 숨긴 1~2개뿐이라 정상이다. 숫자는 가상 데이터 기준이며 실제 효과를 보장하지 않는다.")

fig, ax = plt.subplots(figsize=(6, 3.2))
x = np.arange(len(res)); w = 0.27
ax.bar(x - w, res["Precision@5"], w, label="Precision@5")
ax.bar(x, res["Recall@5"], w, label="Recall@5")
ax.bar(x + w, res["커버리지"], w, label="Coverage")
ax.set_xticks(x); ax.set_xticklabels(["Popularity", "Content", "User CF", "SVD"])
ax.set_title("Recommendation model comparison"); ax.legend()
plt.tight_layout(); plt.show()


---

## 🎯 미션 06. 하이브리드 추천 만들기
**소요시간**: 약 2.0시간

### 📋 미션 배경 및 수행 지침
#### 미션 배경 및 당위성

실제 부대 실무 현장에서는 불명확한 정보 전달이나 수동 업무 처리로 인해 불필요한 행정 소요가 발생하고 주요 공지·규정 확인이 지연되는 문제가 발생하기 쉽습니다. 따라서 **인기도·콘텐츠·유사 장병 점수를 합치는 하이브리드 추천을 만들고 효과를 검증합니다.** 과업을 체계적으로 수행하여 현장 업무 효율성을 극대화하고 정보 누락으로 인한 혼선을 사전에 방지해야 합니다.

이 문제를 해결하기 위해 본 미션에서는 **세 점수를 0~1로 맞춰 가중 합산하고, 장병의 절반으로 가중치 조합을 탐색한 뒤 나머지 절반에서 단일 모델과 비교 수행하고 검증하는** 실습을 단계별로 수행합니다.

학습자는 실습가이드에 제시된 총 4개 세부 활동을 순차적으로 진행하며 핵심 산출물을 도출하고 검증 기준을 달성합니다.

**이번 미션 결과물:** `하이브리드 함수, 가중치 탐색표, 검증 결과 비교표` 완성

### 🔍 평가 및 검증 기준 (PASS 조건)
#### 미션 판정

**평가 구분:** 필수

**핵심 평가 기준:** [필수] 하이브리드·검증 — 하이브리드를 구현하고 탐색용과 검증용을 나누어 효과를 확인했는가? (PASS / FAIL)

#### 미션 PASS 조건

- 요구 산출물(하이브리드 함수, 가중치 탐색표, 검증 결과 비교표)의 완성도 충족


- 원문 대조 점검을 통한 사실 검증 및 정확성 확보


- 교육용 가상 데이터 활용 및 군사보안 수칙 엄수


#### 채점자 판정 메모

> 단순 결과물 생성 여부뿐만 아니라 AI 결과물의 오류를 직접 찾아내고 바로잡았는지 대조표를 확인한다.



In [ ]:
"""인기도·콘텐츠·유사 장병 점수를 가중 합산하는 하이브리드 추천을 만들고, 일부 장병으로 가중치를 고른 뒤 나머지 장병에서 효과를 검증한다."""
import itertools
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import TruncatedSVD

# ---------------------------------------------------------------------------
# [교육용가상자료] 가상 한빛대대 교육 콘텐츠 40개와 가상 장병 60명 (실제 부대·인원과 무관)
# ---------------------------------------------------------------------------
TOPICS = ["AI 기초", "데이터", "보안", "안전", "리더십", "체력", "자격증 대비", "소통·정서"]
LEVELS = {1: "기초", 2: "중급", 3: "심화"}
COURSE_ROWS = [
    ("C01", "인공지능 첫걸음", "AI 기초", 1, 2, "인공지능이 무엇인지 사례로 알아보고 일상과 업무에서 쓰이는 방식을 배웁니다"),
    ("C02", "생성형 AI 안전하게 쓰기", "AI 기초", 1, 1, "생성형 AI에 질문하는 방법과 가상 자료만 입력하는 보안 수칙, 답변 검증 방법을 배웁니다"),
    ("C03", "머신러닝 맛보기", "AI 기초", 2, 3, "데이터로 규칙을 학습하는 머신러닝의 원리와 분류 예측 실습을 해 봅니다"),
    ("C04", "AI 프롬프트 작성 실습", "AI 기초", 2, 2, "목적에 맞는 프롬프트를 작성하고 결과를 비교하며 개선하는 실습을 합니다"),
    ("C05", "AI 윤리와 책임", "AI 기초", 2, 1, "AI 결과의 한계와 편향, 최종 판단은 사람이 한다는 원칙을 사례로 배웁니다"),
    ("C06", "엑셀 데이터 정리 기초", "데이터", 1, 2, "표 데이터를 정리하고 합계와 평균을 구하며 오류를 찾는 방법을 배웁니다"),
    ("C07", "데이터 시각화 입문", "데이터", 1, 2, "막대그래프와 꺾은선 그래프로 데이터를 읽기 쉽게 보여 주는 방법을 배웁니다"),
    ("C08", "파이썬 데이터 분석 기초", "데이터", 2, 4, "파이썬으로 표 데이터를 읽고 통계를 계산하며 그래프를 그리는 실습을 합니다"),
    ("C09", "통계로 읽는 부대 생활", "데이터", 2, 2, "평균과 분포, 상관관계로 생활 데이터를 해석하는 방법을 사례로 배웁니다"),
    ("C10", "데이터 품질 점검 실습", "데이터", 3, 3, "결측과 중복, 형식 오류를 찾아 고치고 데이터 신뢰도를 점검하는 방법을 실습합니다"),
    ("C11", "정보보안 기본 수칙", "보안", 1, 1, "비밀번호 관리와 외부 저장매체 사용 금지, 촬영 금지 등 보안 수칙을 배웁니다"),
    ("C12", "사이버 위협 알아보기", "보안", 1, 2, "악성 메일과 피싱 사례를 보고 의심스러운 연락에 대처하는 방법을 배웁니다"),
    ("C13", "계정과 비밀번호 관리", "보안", 2, 1, "안전한 비밀번호 만들기와 계정 점검 방법을 실습하며 보안 습관을 기릅니다"),
    ("C14", "보안 사고 대응 절차", "보안", 2, 2, "보안 사고가 의심될 때 보고하는 절차와 지휘계통 보고 요령을 사례로 배웁니다"),
    ("C15", "네트워크 보안 이해", "보안", 3, 4, "네트워크 구조와 접근 통제의 원리를 배우고 취약점 점검 개념을 이해합니다"),
    ("C16", "생활관 안전사고 예방", "안전", 1, 1, "전열기 사용과 결빙, 낙상 등 생활관에서 일어나는 안전사고 예방 수칙을 배웁니다"),
    ("C17", "응급처치 기본", "안전", 1, 2, "지혈과 심폐소생술 등 응급처치 기본 동작을 실습하고 보고 요령을 익힙니다"),
    ("C18", "화재 예방과 대피", "안전", 2, 2, "화재 원인과 초기 진화, 대피 경로 확인 방법을 훈련 사례로 배웁니다"),
    ("C19", "동절기 안전 관리", "안전", 2, 1, "한파와 결빙에 대비한 방한 수칙과 동상 예방, 안전 점검 방법을 배웁니다"),
    ("C20", "안전 리더 과정", "안전", 3, 3, "분대 단위 안전 점검표를 만들고 위험 요인을 찾아 개선하는 방법을 실습합니다"),
    ("C21", "분대장 리더십 기초", "리더십", 1, 2, "분대원을 이끄는 기본 자세와 지시, 보고 요령을 사례로 배웁니다"),
    ("C22", "팀 목표 세우기", "리더십", 1, 1, "팀의 목표를 정하고 역할을 나누며 진행 상황을 점검하는 방법을 배웁니다"),
    ("C23", "갈등 조정과 중재", "리더십", 2, 2, "생활관에서 생기는 갈등을 듣고 조정하는 대화 방법을 역할극으로 연습합니다"),
    ("C24", "멘토링과 후임 지도", "리더십", 2, 2, "후임을 지도하고 격려하는 멘토링 방법과 피드백 요령을 사례로 배웁니다"),
    ("C25", "지휘 판단과 의사결정", "리더십", 3, 3, "정보가 부족할 때 근거를 모아 결정하고 책임 있게 보고하는 방법을 사례로 배웁니다"),
    ("C26", "기초 체력 향상 프로그램", "체력", 1, 2, "팔굽혀펴기와 윗몸일으키기, 달리기 기록을 단계별로 올리는 체력 훈련을 합니다"),
    ("C27", "스트레칭과 부상 예방", "체력", 1, 1, "운동 전후 스트레칭과 올바른 자세로 근육 부상을 예방하는 방법을 배웁니다"),
    ("C28", "체력 측정 대비 훈련", "체력", 2, 2, "체력 측정 종목별 요령과 기록 향상 훈련 계획을 세우고 실천합니다"),
    ("C29", "영양과 회복 관리", "체력", 2, 1, "훈련 후 영양 섭취와 수면, 회복 관리로 체력을 유지하는 방법을 배웁니다"),
    ("C30", "전투 체력 심화 과정", "체력", 3, 3, "장거리 달리기와 근지구력을 높이는 심화 체력 훈련과 기록 관리 방법을 배웁니다"),
    ("C31", "컴퓨터활용능력 대비", "자격증 대비", 1, 4, "표 계산과 문서 작성 등 컴퓨터활용능력 시험 문제 풀이와 자격증 대비 요령을 익힙니다"),
    ("C32", "한국사능력 대비", "자격증 대비", 1, 3, "한국사 흐름을 정리하고 기출 문제를 풀며 자격증 시험을 대비합니다"),
    ("C33", "정보처리 기초 대비", "자격증 대비", 2, 4, "자료구조와 알고리즘, 데이터베이스 기초를 정리하고 자격증 시험 문제를 풉니다"),
    ("C34", "데이터 분석 자격 대비", "자격증 대비", 2, 4, "통계와 데이터 처리 개념을 정리하고 데이터 분석 자격증 시험을 대비합니다"),
    ("C35", "전기·정비 기능사 대비", "자격증 대비", 3, 4, "전기 회로와 정비 이론을 정리하고 실기 요령을 익혀 기능사 자격증을 대비합니다"),
    ("C36", "말하기와 발표 기초", "소통·정서", 1, 1, "핵심을 먼저 말하는 보고 방식과 발표 자세를 연습하며 소통 능력을 기릅니다"),
    ("C37", "스트레스 관리와 마음 건강", "소통·정서", 1, 2, "스트레스 신호를 알아차리고 상담 창구를 이용하는 방법 등 마음 건강 관리를 배웁니다"),
    ("C38", "경청과 공감 대화", "소통·정서", 2, 2, "상대의 말을 끝까지 듣고 공감하며 답하는 대화 방법을 짝 연습으로 익힙니다"),
    ("C39", "전역 후 진로 설계", "소통·정서", 2, 2, "전역 후 진로를 정하고 자기개발 계획과 자격증 준비 일정을 세우는 방법을 배웁니다"),
    ("C40", "부대 보고서 글쓰기", "소통·정서", 3, 3, "읽는 사람을 생각해 근거와 결론이 분명한 보고서와 공문을 쓰는 방법을 실습합니다"),
]
courses = pd.DataFrame(COURSE_ROWS, columns=["id", "title", "topic", "level", "hours", "desc"])
CID = list(courses["id"])
RANKS = ["이병", "일병", "상병", "병장"]

# 가상 장병 60명: 계급, 관심 주제 2개, 수강 이력과 평점(1~5)을 난수로 만든다
rng = np.random.default_rng(13)
item_pop = rng.lognormal(0, 0.6, len(courses))
rows, soldier_rows = [], []
for i in range(60):
    sid = f"S{i + 1:02d}"
    rank = RANKS[min(i // 15, 3)]
    interest = list(rng.choice(TOPICS, 2, replace=False))
    max_level = 1 + (RANKS.index(rank) >= 1) + (RANKS.index(rank) >= 3)
    w = np.array([item_pop[j] * (1 + 7 * (courses.topic[j] in interest)) * (1.0 if courses.level[j] <= max_level else 0.3)
                  for j in range(len(courses))])
    n_take = int(rng.integers(7, 14))
    picked = rng.choice(len(courses), n_take, replace=False, p=w / w.sum())
    for j in picked:
        score = 2.9 + 1.3 * (courses.topic[j] in interest) + rng.normal(0, 0.7)
        rows.append((sid, CID[j], int(np.clip(round(score), 1, 5))))
    soldier_rows.append((sid, rank, interest[0], interest[1]))
soldiers = pd.DataFrame(soldier_rows, columns=["id", "rank", "interest1", "interest2"])
ratings = pd.DataFrame(rows, columns=["soldier", "course", "rating"])
SID = list(soldiers["id"])

# 평가용 분리: 평점 4 이상인 수강 중 장병마다 최대 2개를 숨겨 두고(test), 나머지로 추천 모델을 만든다(train)
test_idx = []
for sid, g in ratings[ratings.rating >= 4].groupby("soldier"):
    test_idx += list(rng.choice(g.index, min(2, len(g)), replace=False))
test = ratings.loc[sorted(test_idx)].reset_index(drop=True)
train = ratings.drop(index=test_idx).reset_index(drop=True)
R = train.pivot(index="soldier", columns="course", values="rating").reindex(index=SID, columns=CID)
M = (~R.isna()).values.astype(float)        # 수강 여부(1/0)
RV = R.fillna(0).values                      # 평점 행렬(미수강은 0)
relevant = test.groupby("soldier")["course"].apply(set).to_dict()

# ---- 공통 함수: 점수 계산(인기도·콘텐츠·협업·행렬 분해), 상위 추천, 평가 ----
SIDX = {s: i for i, s in enumerate(SID)}
GLOBAL_MEAN = train["rating"].mean()
_stat = train.groupby("course")["rating"].agg(["mean", "count"]).reindex(CID).fillna({"mean": GLOBAL_MEAN, "count": 0})
POP_SCORE = ((_stat["count"] * _stat["mean"] + 5 * GLOBAL_MEAN) / (_stat["count"] + 5)).values  # 수강 수가 적으면 전체 평균 쪽으로 보정

_vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 3))
TFIDF = _vec.fit_transform(courses["title"] + " " + courses["topic"] + " " + courses["desc"])
ITEM_SIM = cosine_similarity(TFIDF)
USER_SIM = cosine_similarity(RV)
np.fill_diagonal(USER_SIM, 0)
_svd = TruncatedSVD(n_components=6, random_state=13)
_U = _svd.fit_transform(RV)
SVD_SCORE = _U @ _svd.components_


def pop_scores(sid):
    return POP_SCORE.copy()


def content_scores(sid):
    """평점 4 이상으로 수강한 콘텐츠의 설명과 비슷한 콘텐츠일수록 높은 점수"""
    r = RV[SIDX[sid]]
    w = np.where(r >= 4, r - 3, 0)
    if w.sum() == 0:
        return np.zeros(len(CID))
    return (ITEM_SIM * w).sum(axis=1) / w.sum()


def cf_scores(sid, n=10):
    """나와 평점 패턴이 비슷한 장병 n명이 높게 준 콘텐츠일수록 높은 점수"""
    i = SIDX[sid]
    nb = np.argsort(-USER_SIM[i])[:n]
    return (USER_SIM[i, nb][:, None] * RV[nb]).sum(axis=0) / USER_SIM[i, nb].sum()


def svd_scores(sid):
    return SVD_SCORE[SIDX[sid]].copy()


def norm(v, mask):
    """아직 안 들은 콘텐츠들 사이에서 0~1로 맞춘다"""
    x = v.copy()
    lo, hi = x[mask].min(), x[mask].max()
    return np.where(mask, (x - lo) / (hi - lo + 1e-9), 0)


def hybrid_scores(sid, w=(0.2, 0.4, 0.4)):
    unseen = M[SIDX[sid]] == 0
    return (w[0] * norm(pop_scores(sid), unseen) + w[1] * norm(content_scores(sid), unseen)
            + w[2] * norm(cf_scores(sid), unseen))


def top_k(scores, sid, k=5):
    s = np.where(M[SIDX[sid]] == 1, -1e9, scores)    # 이미 들은 콘텐츠는 제외
    return [CID[j] for j in np.argsort(-s)[:k]]


def evaluate(score_fn, k=5, sids=None):
    """장병마다 상위 k개를 추천해 정밀도(Precision@k), 재현율(Recall@k), 커버리지를 구한다."""
    prec, rec, shown = [], [], set()
    for sid, rel in relevant.items():
        if sids is not None and sid not in sids:
            continue
        rc = top_k(score_fn(sid), sid, k)
        hit = len(set(rc) & rel)
        prec.append(hit / k); rec.append(hit / len(rel)); shown |= set(rc)
    return np.mean(prec), np.mean(rec), len(shown) / len(CID)

tune = set(SID[0::2])      # 가중치를 고르는 장병(절반)
check = set(SID[1::2])     # 고른 가중치를 검증하는 장병(나머지 절반)

grid = [(a, b, round(1 - a - b, 1)) for a, b in itertools.product([0, 0.1, 0.2, 0.3], [0.2, 0.3, 0.4, 0.5, 0.6, 0.7])
        if 0 <= round(1 - a - b, 1) <= 0.8]
scored = []
for w in grid:
    p, r, cv = evaluate(lambda s: hybrid_scores(s, w), 5, tune)
    scored.append((p, cv, w))
scored.sort(reverse=True)
print("[가중치 탐색 상위 5] (인기도, 콘텐츠, 유사 장병) - 탐색용 장병 30명 기준")
for p, cv, w in scored[:5]:
    print(f"  {w}: Precision@5 {p:.3f}, 커버리지 {cv:.0%}")
best_w = scored[0][2]
print("선택한 가중치:", best_w)

print("\n[검증용 장병 30명 결과] 가중치를 고를 때 쓰지 않은 장병")
single = {"인기도": pop_scores, "콘텐츠 기반": content_scores, "유사 장병": cf_scores}
table = {}
for name, fn in single.items():
    table[name] = evaluate(fn, 5, check)
table["하이브리드"] = evaluate(lambda s: hybrid_scores(s, best_w), 5, check)
for name, (p, r, cv) in table.items():
    print(f"  {name:<7} Precision@5 {p:.3f}  Recall@5 {r:.3f}  커버리지 {cv:.0%}")

best_single = max(v[0] for k, v in table.items() if k != "하이브리드")
hyb = table["하이브리드"][0]
print(f"\n하이브리드 {hyb:.3f} vs 단일 모델 최고 {best_single:.3f}")
print("하이브리드가 단일 최고 이상:", "목표 달성" if hyb >= best_single else "목표 미달")
print("표본이 30명이라 작은 차이는 우연일 수 있다. 가중치는 정답 없이 데이터에 맞춰 고른 값임을 기억한다.")


---

## 🎯 미션 07. 신규 전입 장병 대응하기
**소요시간**: 약 2.0시간

### 📋 미션 배경 및 수행 지침
#### 미션 배경 및 당위성

실제 부대 실무 현장에서는 불명확한 정보 전달이나 수동 업무 처리로 인해 불필요한 행정 소요가 발생하고 주요 공지·규정 확인이 지연되는 문제가 발생하기 쉽습니다. 따라서 **수강 이력이 없는 신규 전입 장병에게 관심 주제와 계급으로 추천하고, 이력이 쌓이면 개인화로 넘어가게 합니다.** 과업을 체계적으로 수행하여 현장 업무 효율성을 극대화하고 정보 누락으로 인한 혼선을 사전에 방지해야 합니다.

이 문제를 해결하기 위해 본 미션에서는 **관심 주제 일치, 인기도, 난이도 적합을 합친 점수로 신규 전입 장병에게 추천 수행하고 검증하는** 실습을 단계별로 수행합니다.

학습자는 실습가이드에 제시된 총 4개 세부 활동을 순차적으로 진행하며 핵심 산출물을 도출하고 검증 기준을 달성합니다.

**이번 미션 결과물:** `신규 전입 추천 함수, 인기도와의 성능 비교, 이력 누적에 따른 추천 변화` 완성

### 🔍 평가 및 검증 기준 (PASS 조건)
#### 미션 판정

**평가 구분:** 필수

**핵심 평가 기준:** [선택] 신규 전입 대응 — 이력이 없는 장병에게도 근거 있는 추천을 하고 성능을 확인했는가? (PASS / FAIL)

#### 미션 PASS 조건

- 요구 산출물(신규 전입 추천 함수, 인기도와의 성능 비교, 이력 누적에 따른 추천 변화)의 완성도 충족


- 원문 대조 점검을 통한 사실 검증 및 정확성 확보


- 교육용 가상 데이터 활용 및 군사보안 수칙 엄수


#### 채점자 판정 메모

> 단순 결과물 생성 여부뿐만 아니라 AI 결과물의 오류를 직접 찾아내고 바로잡았는지 대조표를 확인한다.



In [ ]:
"""수강 이력이 없는 신규 전입 장병에게 관심 주제·계급·인기도로 추천하고, 이력이 쌓이면 개인화 추천으로 넘어가는 방식을 시험한다."""
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import TruncatedSVD

# ---------------------------------------------------------------------------
# [교육용가상자료] 가상 한빛대대 교육 콘텐츠 40개와 가상 장병 60명 (실제 부대·인원과 무관)
# ---------------------------------------------------------------------------
TOPICS = ["AI 기초", "데이터", "보안", "안전", "리더십", "체력", "자격증 대비", "소통·정서"]
LEVELS = {1: "기초", 2: "중급", 3: "심화"}
COURSE_ROWS = [
    ("C01", "인공지능 첫걸음", "AI 기초", 1, 2, "인공지능이 무엇인지 사례로 알아보고 일상과 업무에서 쓰이는 방식을 배웁니다"),
    ("C02", "생성형 AI 안전하게 쓰기", "AI 기초", 1, 1, "생성형 AI에 질문하는 방법과 가상 자료만 입력하는 보안 수칙, 답변 검증 방법을 배웁니다"),
    ("C03", "머신러닝 맛보기", "AI 기초", 2, 3, "데이터로 규칙을 학습하는 머신러닝의 원리와 분류 예측 실습을 해 봅니다"),
    ("C04", "AI 프롬프트 작성 실습", "AI 기초", 2, 2, "목적에 맞는 프롬프트를 작성하고 결과를 비교하며 개선하는 실습을 합니다"),
    ("C05", "AI 윤리와 책임", "AI 기초", 2, 1, "AI 결과의 한계와 편향, 최종 판단은 사람이 한다는 원칙을 사례로 배웁니다"),
    ("C06", "엑셀 데이터 정리 기초", "데이터", 1, 2, "표 데이터를 정리하고 합계와 평균을 구하며 오류를 찾는 방법을 배웁니다"),
    ("C07", "데이터 시각화 입문", "데이터", 1, 2, "막대그래프와 꺾은선 그래프로 데이터를 읽기 쉽게 보여 주는 방법을 배웁니다"),
    ("C08", "파이썬 데이터 분석 기초", "데이터", 2, 4, "파이썬으로 표 데이터를 읽고 통계를 계산하며 그래프를 그리는 실습을 합니다"),
    ("C09", "통계로 읽는 부대 생활", "데이터", 2, 2, "평균과 분포, 상관관계로 생활 데이터를 해석하는 방법을 사례로 배웁니다"),
    ("C10", "데이터 품질 점검 실습", "데이터", 3, 3, "결측과 중복, 형식 오류를 찾아 고치고 데이터 신뢰도를 점검하는 방법을 실습합니다"),
    ("C11", "정보보안 기본 수칙", "보안", 1, 1, "비밀번호 관리와 외부 저장매체 사용 금지, 촬영 금지 등 보안 수칙을 배웁니다"),
    ("C12", "사이버 위협 알아보기", "보안", 1, 2, "악성 메일과 피싱 사례를 보고 의심스러운 연락에 대처하는 방법을 배웁니다"),
    ("C13", "계정과 비밀번호 관리", "보안", 2, 1, "안전한 비밀번호 만들기와 계정 점검 방법을 실습하며 보안 습관을 기릅니다"),
    ("C14", "보안 사고 대응 절차", "보안", 2, 2, "보안 사고가 의심될 때 보고하는 절차와 지휘계통 보고 요령을 사례로 배웁니다"),
    ("C15", "네트워크 보안 이해", "보안", 3, 4, "네트워크 구조와 접근 통제의 원리를 배우고 취약점 점검 개념을 이해합니다"),
    ("C16", "생활관 안전사고 예방", "안전", 1, 1, "전열기 사용과 결빙, 낙상 등 생활관에서 일어나는 안전사고 예방 수칙을 배웁니다"),
    ("C17", "응급처치 기본", "안전", 1, 2, "지혈과 심폐소생술 등 응급처치 기본 동작을 실습하고 보고 요령을 익힙니다"),
    ("C18", "화재 예방과 대피", "안전", 2, 2, "화재 원인과 초기 진화, 대피 경로 확인 방법을 훈련 사례로 배웁니다"),
    ("C19", "동절기 안전 관리", "안전", 2, 1, "한파와 결빙에 대비한 방한 수칙과 동상 예방, 안전 점검 방법을 배웁니다"),
    ("C20", "안전 리더 과정", "안전", 3, 3, "분대 단위 안전 점검표를 만들고 위험 요인을 찾아 개선하는 방법을 실습합니다"),
    ("C21", "분대장 리더십 기초", "리더십", 1, 2, "분대원을 이끄는 기본 자세와 지시, 보고 요령을 사례로 배웁니다"),
    ("C22", "팀 목표 세우기", "리더십", 1, 1, "팀의 목표를 정하고 역할을 나누며 진행 상황을 점검하는 방법을 배웁니다"),
    ("C23", "갈등 조정과 중재", "리더십", 2, 2, "생활관에서 생기는 갈등을 듣고 조정하는 대화 방법을 역할극으로 연습합니다"),
    ("C24", "멘토링과 후임 지도", "리더십", 2, 2, "후임을 지도하고 격려하는 멘토링 방법과 피드백 요령을 사례로 배웁니다"),
    ("C25", "지휘 판단과 의사결정", "리더십", 3, 3, "정보가 부족할 때 근거를 모아 결정하고 책임 있게 보고하는 방법을 사례로 배웁니다"),
    ("C26", "기초 체력 향상 프로그램", "체력", 1, 2, "팔굽혀펴기와 윗몸일으키기, 달리기 기록을 단계별로 올리는 체력 훈련을 합니다"),
    ("C27", "스트레칭과 부상 예방", "체력", 1, 1, "운동 전후 스트레칭과 올바른 자세로 근육 부상을 예방하는 방법을 배웁니다"),
    ("C28", "체력 측정 대비 훈련", "체력", 2, 2, "체력 측정 종목별 요령과 기록 향상 훈련 계획을 세우고 실천합니다"),
    ("C29", "영양과 회복 관리", "체력", 2, 1, "훈련 후 영양 섭취와 수면, 회복 관리로 체력을 유지하는 방법을 배웁니다"),
    ("C30", "전투 체력 심화 과정", "체력", 3, 3, "장거리 달리기와 근지구력을 높이는 심화 체력 훈련과 기록 관리 방법을 배웁니다"),
    ("C31", "컴퓨터활용능력 대비", "자격증 대비", 1, 4, "표 계산과 문서 작성 등 컴퓨터활용능력 시험 문제 풀이와 자격증 대비 요령을 익힙니다"),
    ("C32", "한국사능력 대비", "자격증 대비", 1, 3, "한국사 흐름을 정리하고 기출 문제를 풀며 자격증 시험을 대비합니다"),
    ("C33", "정보처리 기초 대비", "자격증 대비", 2, 4, "자료구조와 알고리즘, 데이터베이스 기초를 정리하고 자격증 시험 문제를 풉니다"),
    ("C34", "데이터 분석 자격 대비", "자격증 대비", 2, 4, "통계와 데이터 처리 개념을 정리하고 데이터 분석 자격증 시험을 대비합니다"),
    ("C35", "전기·정비 기능사 대비", "자격증 대비", 3, 4, "전기 회로와 정비 이론을 정리하고 실기 요령을 익혀 기능사 자격증을 대비합니다"),
    ("C36", "말하기와 발표 기초", "소통·정서", 1, 1, "핵심을 먼저 말하는 보고 방식과 발표 자세를 연습하며 소통 능력을 기릅니다"),
    ("C37", "스트레스 관리와 마음 건강", "소통·정서", 1, 2, "스트레스 신호를 알아차리고 상담 창구를 이용하는 방법 등 마음 건강 관리를 배웁니다"),
    ("C38", "경청과 공감 대화", "소통·정서", 2, 2, "상대의 말을 끝까지 듣고 공감하며 답하는 대화 방법을 짝 연습으로 익힙니다"),
    ("C39", "전역 후 진로 설계", "소통·정서", 2, 2, "전역 후 진로를 정하고 자기개발 계획과 자격증 준비 일정을 세우는 방법을 배웁니다"),
    ("C40", "부대 보고서 글쓰기", "소통·정서", 3, 3, "읽는 사람을 생각해 근거와 결론이 분명한 보고서와 공문을 쓰는 방법을 실습합니다"),
]
courses = pd.DataFrame(COURSE_ROWS, columns=["id", "title", "topic", "level", "hours", "desc"])
CID = list(courses["id"])
RANKS = ["이병", "일병", "상병", "병장"]

# 가상 장병 60명: 계급, 관심 주제 2개, 수강 이력과 평점(1~5)을 난수로 만든다
rng = np.random.default_rng(13)
item_pop = rng.lognormal(0, 0.6, len(courses))
rows, soldier_rows = [], []
for i in range(60):
    sid = f"S{i + 1:02d}"
    rank = RANKS[min(i // 15, 3)]
    interest = list(rng.choice(TOPICS, 2, replace=False))
    max_level = 1 + (RANKS.index(rank) >= 1) + (RANKS.index(rank) >= 3)
    w = np.array([item_pop[j] * (1 + 7 * (courses.topic[j] in interest)) * (1.0 if courses.level[j] <= max_level else 0.3)
                  for j in range(len(courses))])
    n_take = int(rng.integers(7, 14))
    picked = rng.choice(len(courses), n_take, replace=False, p=w / w.sum())
    for j in picked:
        score = 2.9 + 1.3 * (courses.topic[j] in interest) + rng.normal(0, 0.7)
        rows.append((sid, CID[j], int(np.clip(round(score), 1, 5))))
    soldier_rows.append((sid, rank, interest[0], interest[1]))
soldiers = pd.DataFrame(soldier_rows, columns=["id", "rank", "interest1", "interest2"])
ratings = pd.DataFrame(rows, columns=["soldier", "course", "rating"])
SID = list(soldiers["id"])

# 평가용 분리: 평점 4 이상인 수강 중 장병마다 최대 2개를 숨겨 두고(test), 나머지로 추천 모델을 만든다(train)
test_idx = []
for sid, g in ratings[ratings.rating >= 4].groupby("soldier"):
    test_idx += list(rng.choice(g.index, min(2, len(g)), replace=False))
test = ratings.loc[sorted(test_idx)].reset_index(drop=True)
train = ratings.drop(index=test_idx).reset_index(drop=True)
R = train.pivot(index="soldier", columns="course", values="rating").reindex(index=SID, columns=CID)
M = (~R.isna()).values.astype(float)        # 수강 여부(1/0)
RV = R.fillna(0).values                      # 평점 행렬(미수강은 0)
relevant = test.groupby("soldier")["course"].apply(set).to_dict()

# ---- 공통 함수: 점수 계산(인기도·콘텐츠·협업·행렬 분해), 상위 추천, 평가 ----
SIDX = {s: i for i, s in enumerate(SID)}
GLOBAL_MEAN = train["rating"].mean()
_stat = train.groupby("course")["rating"].agg(["mean", "count"]).reindex(CID).fillna({"mean": GLOBAL_MEAN, "count": 0})
POP_SCORE = ((_stat["count"] * _stat["mean"] + 5 * GLOBAL_MEAN) / (_stat["count"] + 5)).values  # 수강 수가 적으면 전체 평균 쪽으로 보정

_vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 3))
TFIDF = _vec.fit_transform(courses["title"] + " " + courses["topic"] + " " + courses["desc"])
ITEM_SIM = cosine_similarity(TFIDF)
USER_SIM = cosine_similarity(RV)
np.fill_diagonal(USER_SIM, 0)
_svd = TruncatedSVD(n_components=6, random_state=13)
_U = _svd.fit_transform(RV)
SVD_SCORE = _U @ _svd.components_


def pop_scores(sid):
    return POP_SCORE.copy()


def content_scores(sid):
    """평점 4 이상으로 수강한 콘텐츠의 설명과 비슷한 콘텐츠일수록 높은 점수"""
    r = RV[SIDX[sid]]
    w = np.where(r >= 4, r - 3, 0)
    if w.sum() == 0:
        return np.zeros(len(CID))
    return (ITEM_SIM * w).sum(axis=1) / w.sum()


def cf_scores(sid, n=10):
    """나와 평점 패턴이 비슷한 장병 n명이 높게 준 콘텐츠일수록 높은 점수"""
    i = SIDX[sid]
    nb = np.argsort(-USER_SIM[i])[:n]
    return (USER_SIM[i, nb][:, None] * RV[nb]).sum(axis=0) / USER_SIM[i, nb].sum()


def svd_scores(sid):
    return SVD_SCORE[SIDX[sid]].copy()


def norm(v, mask):
    """아직 안 들은 콘텐츠들 사이에서 0~1로 맞춘다"""
    x = v.copy()
    lo, hi = x[mask].min(), x[mask].max()
    return np.where(mask, (x - lo) / (hi - lo + 1e-9), 0)


def hybrid_scores(sid, w=(0.2, 0.4, 0.4)):
    unseen = M[SIDX[sid]] == 0
    return (w[0] * norm(pop_scores(sid), unseen) + w[1] * norm(content_scores(sid), unseen)
            + w[2] * norm(cf_scores(sid), unseen))


def top_k(scores, sid, k=5):
    s = np.where(M[SIDX[sid]] == 1, -1e9, scores)    # 이미 들은 콘텐츠는 제외
    return [CID[j] for j in np.argsort(-s)[:k]]


def evaluate(score_fn, k=5, sids=None):
    """장병마다 상위 k개를 추천해 정밀도(Precision@k), 재현율(Recall@k), 커버리지를 구한다."""
    prec, rec, shown = [], [], set()
    for sid, rel in relevant.items():
        if sids is not None and sid not in sids:
            continue
        rc = top_k(score_fn(sid), sid, k)
        hit = len(set(rc) & rel)
        prec.append(hit / k); rec.append(hit / len(rel)); shown |= set(rc)
    return np.mean(prec), np.mean(rec), len(shown) / len(CID)

# ---- 신규 전입 장병(수강 이력 없음) 대응 ----
MAXLEVEL = {"이병": 1, "일병": 2, "상병": 2, "병장": 3}


def cold_scores(interests, rank):
    """이력이 없을 때는 관심 주제 일치(60%) + 인기도(30%) + 난이도 적합(10%)으로 점수를 매긴다."""
    topic_hit = courses["topic"].isin(interests).values.astype(float)
    level_fit = (courses["level"].values <= MAXLEVEL[rank]).astype(float)
    pop = (POP_SCORE - POP_SCORE.min()) / (POP_SCORE.max() - POP_SCORE.min())
    return 0.6 * topic_hit + 0.3 * pop + 0.1 * level_fit


def top_k_new(scores, k=5, exclude=()):
    s = scores.copy()
    for j, c in enumerate(CID):
        if c in exclude:
            s[j] = -1e9
    return [CID[j] for j in np.argsort(-s)[:k]]

new = {"관심": ["보안", "자격증 대비"], "계급": "이병"}
sc = cold_scores(new["관심"], new["계급"])
print(f"신규 전입 장병 (계급 {new['계급']}, 관심 {', '.join(new['관심'])}) - 수강 이력 0건")
for c in top_k_new(sc):
    j = CID.index(c)
    print(f"  {c} {courses.title[j]} [{courses.topic[j]}, {LEVELS[courses.level[j]]}]")

# 평가: 기존 장병을 '이력 없는 신규 장병'처럼 다루어 관심 주제만으로 추천해 보고, 숨긴 정답을 맞히는지 본다
def eval_cold(fn):
    hits, shown = 0, set()
    for sid, rel in relevant.items():
        me = soldiers[soldiers.id == sid].iloc[0]
        rc = top_k_new(fn(me), 5, exclude=set(train.loc[train.soldier == sid, "course"]))
        hits += len(set(rc) & rel); shown |= set(rc)
    return hits / (5 * len(relevant)), len(shown) / len(CID)


p_cold, cv_cold = eval_cold(lambda me: cold_scores([me["interest1"], me["interest2"]], me["rank"]))
p_pop, cv_pop = eval_cold(lambda me: POP_SCORE)
p_hyb = evaluate(hybrid_scores, 5)[0]
print(f"\n[이력이 없을 때] 인기도만: Precision@5 {p_pop:.3f}, 커버리지 {cv_pop:.0%}")
print(f"[이력이 없을 때] 관심 주제+계급+인기도: Precision@5 {p_cold:.3f}, 커버리지 {cv_cold:.0%}")
print(f"[이력이 있을 때] 하이브리드: Precision@5 {p_hyb:.3f}")
print("(가상 데이터는 관심 주제가 수강 이력을 만들도록 설계되어 관심 주제가 정확히 입력된 경우에 유리하다. 실제로는 입력한 관심이 부정확할 수 있다.)")

# 이력이 1~3개 쌓이면 콘텐츠 기반을 섞는다: 새 장병이 첫 수강 2개에 높은 평점을 준 상황
def warm_scores(first_taken, interests, rank, n_hist):
    w = np.zeros(len(CID))
    for c in first_taken:
        w[CID.index(c)] = 2
    content = (ITEM_SIM * w).sum(axis=1) / w.sum()
    cold = cold_scores(interests, rank)
    mix = min(n_hist / 3, 1)             # 이력 3개가 되면 개인화 비중 100%
    return (1 - mix) * cold + mix * (content / (content.max() + 1e-9))


first = ["C11", "C12"]
for n_hist in [0, 1, 2, 3]:
    s = warm_scores(first[:max(n_hist, 1)], new["관심"], new["계급"], n_hist)
    print(f"이력 {n_hist}개 후 추천:", top_k_new(s, 5, exclude=set(first[:n_hist])))

ok = p_cold > p_pop
print("\n신규 전입 대응이 인기도만 쓸 때보다 정확:", "목표 달성" if ok else "목표 미달")
print("신규 전입자에게는 관심 주제를 직접 물어보고, 이력이 쌓이면 개인화로 전환한다. 입력 정보는 본인이 알려 준 범위로 제한한다.")


---

## 🎯 미션 08. 추천 이유 설명하기
**소요시간**: 약 2.0시간

### 📋 미션 배경 및 수행 지침
#### 미션 배경 및 당위성

실제 부대 실무 현장에서는 불명확한 정보 전달이나 수동 업무 처리로 인해 불필요한 행정 소요가 발생하고 주요 공지·규정 확인이 지연되는 문제가 발생하기 쉽습니다. 따라서 **추천된 콘텐츠마다 사람이 이해할 수 있는 추천 이유를 만들어 보여 줍니다.** 과업을 체계적으로 수행하여 현장 업무 효율성을 극대화하고 정보 누락으로 인한 혼선을 사전에 방지해야 합니다.

이 문제를 해결하기 위해 본 미션에서는 **관심 주제 일치, 높게 평가한 콘텐츠와의 설명 유사도, 유사 장병 평가 인원수, 계급에 맞는 난이도를 근거로 이유 문장을 제작하고 안정성을 확보하는** 실습을 단계별로 수행합니다.

학습자는 실습가이드에 제시된 총 4개 세부 활동을 순차적으로 진행하며 핵심 산출물을 도출하고 검증 기준을 달성합니다.

**이번 미션 결과물:** `추천 이유 생성 함수, 장병별 추천 카드, 설명 점검 결과` 완성

### 🔍 평가 및 검증 기준 (PASS 조건)
#### 미션 판정

**평가 구분:** 필수

**핵심 평가 기준:** [필수] 추천 이유 설명 — 모든 추천에 근거 있는 이유를 붙이고 다른 장병 정보가 노출되지 않음을 확인했는가? (PASS / FAIL)

#### 미션 PASS 조건

- 요구 산출물(추천 이유 생성 함수, 장병별 추천 카드, 설명 점검 결과)의 완성도 충족


- 원문 대조 점검을 통한 사실 검증 및 정확성 확보


- 교육용 가상 데이터 활용 및 군사보안 수칙 엄수


#### 채점자 판정 메모

> 단순 결과물 생성 여부뿐만 아니라 AI 결과물의 오류를 직접 찾아내고 바로잡았는지 대조표를 확인한다.



In [ ]:
"""추천된 콘텐츠마다 관심 주제 일치·비슷한 수강 콘텐츠·유사 장병 평가·난이도 적합을 근거로 추천 이유를 문장으로 설명한다."""
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import TruncatedSVD

# ---------------------------------------------------------------------------
# [교육용가상자료] 가상 한빛대대 교육 콘텐츠 40개와 가상 장병 60명 (실제 부대·인원과 무관)
# ---------------------------------------------------------------------------
TOPICS = ["AI 기초", "데이터", "보안", "안전", "리더십", "체력", "자격증 대비", "소통·정서"]
LEVELS = {1: "기초", 2: "중급", 3: "심화"}
COURSE_ROWS = [
    ("C01", "인공지능 첫걸음", "AI 기초", 1, 2, "인공지능이 무엇인지 사례로 알아보고 일상과 업무에서 쓰이는 방식을 배웁니다"),
    ("C02", "생성형 AI 안전하게 쓰기", "AI 기초", 1, 1, "생성형 AI에 질문하는 방법과 가상 자료만 입력하는 보안 수칙, 답변 검증 방법을 배웁니다"),
    ("C03", "머신러닝 맛보기", "AI 기초", 2, 3, "데이터로 규칙을 학습하는 머신러닝의 원리와 분류 예측 실습을 해 봅니다"),
    ("C04", "AI 프롬프트 작성 실습", "AI 기초", 2, 2, "목적에 맞는 프롬프트를 작성하고 결과를 비교하며 개선하는 실습을 합니다"),
    ("C05", "AI 윤리와 책임", "AI 기초", 2, 1, "AI 결과의 한계와 편향, 최종 판단은 사람이 한다는 원칙을 사례로 배웁니다"),
    ("C06", "엑셀 데이터 정리 기초", "데이터", 1, 2, "표 데이터를 정리하고 합계와 평균을 구하며 오류를 찾는 방법을 배웁니다"),
    ("C07", "데이터 시각화 입문", "데이터", 1, 2, "막대그래프와 꺾은선 그래프로 데이터를 읽기 쉽게 보여 주는 방법을 배웁니다"),
    ("C08", "파이썬 데이터 분석 기초", "데이터", 2, 4, "파이썬으로 표 데이터를 읽고 통계를 계산하며 그래프를 그리는 실습을 합니다"),
    ("C09", "통계로 읽는 부대 생활", "데이터", 2, 2, "평균과 분포, 상관관계로 생활 데이터를 해석하는 방법을 사례로 배웁니다"),
    ("C10", "데이터 품질 점검 실습", "데이터", 3, 3, "결측과 중복, 형식 오류를 찾아 고치고 데이터 신뢰도를 점검하는 방법을 실습합니다"),
    ("C11", "정보보안 기본 수칙", "보안", 1, 1, "비밀번호 관리와 외부 저장매체 사용 금지, 촬영 금지 등 보안 수칙을 배웁니다"),
    ("C12", "사이버 위협 알아보기", "보안", 1, 2, "악성 메일과 피싱 사례를 보고 의심스러운 연락에 대처하는 방법을 배웁니다"),
    ("C13", "계정과 비밀번호 관리", "보안", 2, 1, "안전한 비밀번호 만들기와 계정 점검 방법을 실습하며 보안 습관을 기릅니다"),
    ("C14", "보안 사고 대응 절차", "보안", 2, 2, "보안 사고가 의심될 때 보고하는 절차와 지휘계통 보고 요령을 사례로 배웁니다"),
    ("C15", "네트워크 보안 이해", "보안", 3, 4, "네트워크 구조와 접근 통제의 원리를 배우고 취약점 점검 개념을 이해합니다"),
    ("C16", "생활관 안전사고 예방", "안전", 1, 1, "전열기 사용과 결빙, 낙상 등 생활관에서 일어나는 안전사고 예방 수칙을 배웁니다"),
    ("C17", "응급처치 기본", "안전", 1, 2, "지혈과 심폐소생술 등 응급처치 기본 동작을 실습하고 보고 요령을 익힙니다"),
    ("C18", "화재 예방과 대피", "안전", 2, 2, "화재 원인과 초기 진화, 대피 경로 확인 방법을 훈련 사례로 배웁니다"),
    ("C19", "동절기 안전 관리", "안전", 2, 1, "한파와 결빙에 대비한 방한 수칙과 동상 예방, 안전 점검 방법을 배웁니다"),
    ("C20", "안전 리더 과정", "안전", 3, 3, "분대 단위 안전 점검표를 만들고 위험 요인을 찾아 개선하는 방법을 실습합니다"),
    ("C21", "분대장 리더십 기초", "리더십", 1, 2, "분대원을 이끄는 기본 자세와 지시, 보고 요령을 사례로 배웁니다"),
    ("C22", "팀 목표 세우기", "리더십", 1, 1, "팀의 목표를 정하고 역할을 나누며 진행 상황을 점검하는 방법을 배웁니다"),
    ("C23", "갈등 조정과 중재", "리더십", 2, 2, "생활관에서 생기는 갈등을 듣고 조정하는 대화 방법을 역할극으로 연습합니다"),
    ("C24", "멘토링과 후임 지도", "리더십", 2, 2, "후임을 지도하고 격려하는 멘토링 방법과 피드백 요령을 사례로 배웁니다"),
    ("C25", "지휘 판단과 의사결정", "리더십", 3, 3, "정보가 부족할 때 근거를 모아 결정하고 책임 있게 보고하는 방법을 사례로 배웁니다"),
    ("C26", "기초 체력 향상 프로그램", "체력", 1, 2, "팔굽혀펴기와 윗몸일으키기, 달리기 기록을 단계별로 올리는 체력 훈련을 합니다"),
    ("C27", "스트레칭과 부상 예방", "체력", 1, 1, "운동 전후 스트레칭과 올바른 자세로 근육 부상을 예방하는 방법을 배웁니다"),
    ("C28", "체력 측정 대비 훈련", "체력", 2, 2, "체력 측정 종목별 요령과 기록 향상 훈련 계획을 세우고 실천합니다"),
    ("C29", "영양과 회복 관리", "체력", 2, 1, "훈련 후 영양 섭취와 수면, 회복 관리로 체력을 유지하는 방법을 배웁니다"),
    ("C30", "전투 체력 심화 과정", "체력", 3, 3, "장거리 달리기와 근지구력을 높이는 심화 체력 훈련과 기록 관리 방법을 배웁니다"),
    ("C31", "컴퓨터활용능력 대비", "자격증 대비", 1, 4, "표 계산과 문서 작성 등 컴퓨터활용능력 시험 문제 풀이와 자격증 대비 요령을 익힙니다"),
    ("C32", "한국사능력 대비", "자격증 대비", 1, 3, "한국사 흐름을 정리하고 기출 문제를 풀며 자격증 시험을 대비합니다"),
    ("C33", "정보처리 기초 대비", "자격증 대비", 2, 4, "자료구조와 알고리즘, 데이터베이스 기초를 정리하고 자격증 시험 문제를 풉니다"),
    ("C34", "데이터 분석 자격 대비", "자격증 대비", 2, 4, "통계와 데이터 처리 개념을 정리하고 데이터 분석 자격증 시험을 대비합니다"),
    ("C35", "전기·정비 기능사 대비", "자격증 대비", 3, 4, "전기 회로와 정비 이론을 정리하고 실기 요령을 익혀 기능사 자격증을 대비합니다"),
    ("C36", "말하기와 발표 기초", "소통·정서", 1, 1, "핵심을 먼저 말하는 보고 방식과 발표 자세를 연습하며 소통 능력을 기릅니다"),
    ("C37", "스트레스 관리와 마음 건강", "소통·정서", 1, 2, "스트레스 신호를 알아차리고 상담 창구를 이용하는 방법 등 마음 건강 관리를 배웁니다"),
    ("C38", "경청과 공감 대화", "소통·정서", 2, 2, "상대의 말을 끝까지 듣고 공감하며 답하는 대화 방법을 짝 연습으로 익힙니다"),
    ("C39", "전역 후 진로 설계", "소통·정서", 2, 2, "전역 후 진로를 정하고 자기개발 계획과 자격증 준비 일정을 세우는 방법을 배웁니다"),
    ("C40", "부대 보고서 글쓰기", "소통·정서", 3, 3, "읽는 사람을 생각해 근거와 결론이 분명한 보고서와 공문을 쓰는 방법을 실습합니다"),
]
courses = pd.DataFrame(COURSE_ROWS, columns=["id", "title", "topic", "level", "hours", "desc"])
CID = list(courses["id"])
RANKS = ["이병", "일병", "상병", "병장"]

# 가상 장병 60명: 계급, 관심 주제 2개, 수강 이력과 평점(1~5)을 난수로 만든다
rng = np.random.default_rng(13)
item_pop = rng.lognormal(0, 0.6, len(courses))
rows, soldier_rows = [], []
for i in range(60):
    sid = f"S{i + 1:02d}"
    rank = RANKS[min(i // 15, 3)]
    interest = list(rng.choice(TOPICS, 2, replace=False))
    max_level = 1 + (RANKS.index(rank) >= 1) + (RANKS.index(rank) >= 3)
    w = np.array([item_pop[j] * (1 + 7 * (courses.topic[j] in interest)) * (1.0 if courses.level[j] <= max_level else 0.3)
                  for j in range(len(courses))])
    n_take = int(rng.integers(7, 14))
    picked = rng.choice(len(courses), n_take, replace=False, p=w / w.sum())
    for j in picked:
        score = 2.9 + 1.3 * (courses.topic[j] in interest) + rng.normal(0, 0.7)
        rows.append((sid, CID[j], int(np.clip(round(score), 1, 5))))
    soldier_rows.append((sid, rank, interest[0], interest[1]))
soldiers = pd.DataFrame(soldier_rows, columns=["id", "rank", "interest1", "interest2"])
ratings = pd.DataFrame(rows, columns=["soldier", "course", "rating"])
SID = list(soldiers["id"])

# 평가용 분리: 평점 4 이상인 수강 중 장병마다 최대 2개를 숨겨 두고(test), 나머지로 추천 모델을 만든다(train)
test_idx = []
for sid, g in ratings[ratings.rating >= 4].groupby("soldier"):
    test_idx += list(rng.choice(g.index, min(2, len(g)), replace=False))
test = ratings.loc[sorted(test_idx)].reset_index(drop=True)
train = ratings.drop(index=test_idx).reset_index(drop=True)
R = train.pivot(index="soldier", columns="course", values="rating").reindex(index=SID, columns=CID)
M = (~R.isna()).values.astype(float)        # 수강 여부(1/0)
RV = R.fillna(0).values                      # 평점 행렬(미수강은 0)
relevant = test.groupby("soldier")["course"].apply(set).to_dict()

# ---- 공통 함수: 점수 계산(인기도·콘텐츠·협업·행렬 분해), 상위 추천, 평가 ----
SIDX = {s: i for i, s in enumerate(SID)}
GLOBAL_MEAN = train["rating"].mean()
_stat = train.groupby("course")["rating"].agg(["mean", "count"]).reindex(CID).fillna({"mean": GLOBAL_MEAN, "count": 0})
POP_SCORE = ((_stat["count"] * _stat["mean"] + 5 * GLOBAL_MEAN) / (_stat["count"] + 5)).values  # 수강 수가 적으면 전체 평균 쪽으로 보정

_vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 3))
TFIDF = _vec.fit_transform(courses["title"] + " " + courses["topic"] + " " + courses["desc"])
ITEM_SIM = cosine_similarity(TFIDF)
USER_SIM = cosine_similarity(RV)
np.fill_diagonal(USER_SIM, 0)
_svd = TruncatedSVD(n_components=6, random_state=13)
_U = _svd.fit_transform(RV)
SVD_SCORE = _U @ _svd.components_


def pop_scores(sid):
    return POP_SCORE.copy()


def content_scores(sid):
    """평점 4 이상으로 수강한 콘텐츠의 설명과 비슷한 콘텐츠일수록 높은 점수"""
    r = RV[SIDX[sid]]
    w = np.where(r >= 4, r - 3, 0)
    if w.sum() == 0:
        return np.zeros(len(CID))
    return (ITEM_SIM * w).sum(axis=1) / w.sum()


def cf_scores(sid, n=10):
    """나와 평점 패턴이 비슷한 장병 n명이 높게 준 콘텐츠일수록 높은 점수"""
    i = SIDX[sid]
    nb = np.argsort(-USER_SIM[i])[:n]
    return (USER_SIM[i, nb][:, None] * RV[nb]).sum(axis=0) / USER_SIM[i, nb].sum()


def svd_scores(sid):
    return SVD_SCORE[SIDX[sid]].copy()


def norm(v, mask):
    """아직 안 들은 콘텐츠들 사이에서 0~1로 맞춘다"""
    x = v.copy()
    lo, hi = x[mask].min(), x[mask].max()
    return np.where(mask, (x - lo) / (hi - lo + 1e-9), 0)


def hybrid_scores(sid, w=(0.2, 0.4, 0.4)):
    unseen = M[SIDX[sid]] == 0
    return (w[0] * norm(pop_scores(sid), unseen) + w[1] * norm(content_scores(sid), unseen)
            + w[2] * norm(cf_scores(sid), unseen))


def top_k(scores, sid, k=5):
    s = np.where(M[SIDX[sid]] == 1, -1e9, scores)    # 이미 들은 콘텐츠는 제외
    return [CID[j] for j in np.argsort(-s)[:k]]


def evaluate(score_fn, k=5, sids=None):
    """장병마다 상위 k개를 추천해 정밀도(Precision@k), 재현율(Recall@k), 커버리지를 구한다."""
    prec, rec, shown = [], [], set()
    for sid, rel in relevant.items():
        if sids is not None and sid not in sids:
            continue
        rc = top_k(score_fn(sid), sid, k)
        hit = len(set(rc) & rel)
        prec.append(hit / k); rec.append(hit / len(rel)); shown |= set(rc)
    return np.mean(prec), np.mean(rec), len(shown) / len(CID)

# ---- 신규 전입 장병(수강 이력 없음) 대응 ----
MAXLEVEL = {"이병": 1, "일병": 2, "상병": 2, "병장": 3}


def cold_scores(interests, rank):
    """이력이 없을 때는 관심 주제 일치(60%) + 인기도(30%) + 난이도 적합(10%)으로 점수를 매긴다."""
    topic_hit = courses["topic"].isin(interests).values.astype(float)
    level_fit = (courses["level"].values <= MAXLEVEL[rank]).astype(float)
    pop = (POP_SCORE - POP_SCORE.min()) / (POP_SCORE.max() - POP_SCORE.min())
    return 0.6 * topic_hit + 0.3 * pop + 0.1 * level_fit


def top_k_new(scores, k=5, exclude=()):
    s = scores.copy()
    for j, c in enumerate(CID):
        if c in exclude:
            s[j] = -1e9
    return [CID[j] for j in np.argsort(-s)[:k]]

# ---- 추천 이유 설명 ----
def explain(sid, cid, n=10):
    """추천 이유를 문장으로 만든다. 다른 장병의 이름은 밝히지 않고 인원수만 말한다."""
    i, j = SIDX[sid], CID.index(cid)
    c, me = courses.iloc[j], soldiers.iloc[i]
    why = []
    if c["topic"] in (me["interest1"], me["interest2"]):
        why.append(f"관심 주제('{c['topic']}')에 해당합니다")
    liked = [k for k in range(len(CID)) if RV[i, k] >= 4]
    if liked:
        k = max(liked, key=lambda x: ITEM_SIM[j, x])
        if ITEM_SIM[j, k] >= 0.15:
            why.append(f"높게 평가한 콘텐츠('{courses.title[k]}')와 설명이 비슷합니다(유사도 {ITEM_SIM[j, k]:.2f})")
    nb = np.argsort(-USER_SIM[i])[:n]
    fans = [x for x in nb if RV[x, j] >= 4]
    if fans:
        why.append(f"수강 패턴이 비슷한 장병 {n}명 중 {len(fans)}명이 높게 평가했습니다")
    if c["level"] <= MAXLEVEL[me["rank"]]:
        why.append(f"{me['rank']} 수준에 맞는 {LEVELS[c['level']]} 과정입니다")
    if not why:
        why.append("전체 장병에게 인기가 높은 과정입니다")
    return why

for sid in ["S01", "S31", "S52"]:
    me = soldiers[soldiers.id == sid].iloc[0]
    print(f"\n=== {sid} ({me['rank']}, 관심 {me['interest1']}·{me['interest2']}) 추천과 이유 ===")
    for c in top_k(hybrid_scores(sid), sid, 3):
        print(f"- {c} {courses.title[CID.index(c)]} ({courses.hours[CID.index(c)]}시간)")
        for r in explain(sid, c):
            print(f"    * {r}")

# 설명 품질 점검
n_rec = n_with = n_leak = 0
for sid in SID:
    for c in top_k(hybrid_scores(sid), sid, 5):
        why = explain(sid, c)
        n_rec += 1
        n_with += len(why) >= 1
        n_leak += any(s in " ".join(why) for s in SID)       # 다른 장병 ID가 문장에 들어갔는지
print(f"\n추천 {n_rec}건 중 이유가 있는 추천 {n_with}건 ({n_with / n_rec:.0%})")
print(f"다른 장병 ID가 노출된 설명: {n_leak}건")
print("개인정보 보호:", "목표 달성" if n_leak == 0 else "목표 미달")
cnt = pd.Series([len(explain(s, c)) for s in SID for c in top_k(hybrid_scores(s), s, 5)]).value_counts().sort_index()
print("추천 1건당 제시한 이유 개수 분포:", cnt.to_dict())
print("\n추천 이유는 판단을 돕는 근거일 뿐이며, 이수 여부는 본인과 지휘관이 결정한다.")


---

## 🎯 미션 09. 편향·다양성 점검하기
**소요시간**: 약 2.0시간

### 📋 미션 배경 및 수행 지침
#### 미션 배경 및 당위성

실제 부대 실무 현장에서는 불명확한 정보 전달이나 수동 업무 처리로 인해 불필요한 행정 소요가 발생하고 주요 공지·규정 확인이 지연되는 문제가 발생하기 쉽습니다. 따라서 **추천이 같은 주제나 인기 콘텐츠에 쏠리지 않는지 점검하고 다양성을 높입니다.** 과업을 체계적으로 수행하여 현장 업무 효율성을 극대화하고 정보 누락으로 인한 혼선을 사전에 방지해야 합니다.

이 문제를 해결하기 위해 본 미션에서는 **추천 목록의 주제 수, 목록 안 다양도, 관심 주제 비율, 자주 노출되는 콘텐츠, 계급별 난이도를 계산해 쏠림을 찾습니다 과정을 진행하는** 실습을 단계별로 수행합니다.

학습자는 실습가이드에 제시된 총 4개 세부 활동을 순차적으로 진행하며 핵심 산출물을 도출하고 검증 기준을 달성합니다.

**이번 미션 결과물:** `편향·다양성 지표표, 다시 정렬 함수, 전후 비교와 해석` 완성

### 🔍 평가 및 검증 기준 (PASS 조건)
#### 미션 판정

**평가 구분:** 필수

**핵심 평가 기준:** [필수] 편향·다양성 점검 — 쏠림을 지표로 확인하고 다양성 개선과 정확도 변화를 함께 비교했는가? (PASS / FAIL)

#### 미션 PASS 조건

- 요구 산출물(편향·다양성 지표표, 다시 정렬 함수, 전후 비교와 해석)의 완성도 충족


- 원문 대조 점검을 통한 사실 검증 및 정확성 확보


- 교육용 가상 데이터 활용 및 군사보안 수칙 엄수


#### 채점자 판정 메모

> 단순 결과물 생성 여부뿐만 아니라 AI 결과물의 오류를 직접 찾아내고 바로잡았는지 대조표를 확인한다.



In [ ]:
"""추천 결과가 같은 주제나 인기 콘텐츠에 쏠리지 않는지 점검하고, 주제별 상한을 두는 다시 정렬로 다양성을 높였을 때 정확도 변화를 비교한다."""
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import TruncatedSVD

# ---------------------------------------------------------------------------
# [교육용가상자료] 가상 한빛대대 교육 콘텐츠 40개와 가상 장병 60명 (실제 부대·인원과 무관)
# ---------------------------------------------------------------------------
TOPICS = ["AI 기초", "데이터", "보안", "안전", "리더십", "체력", "자격증 대비", "소통·정서"]
LEVELS = {1: "기초", 2: "중급", 3: "심화"}
COURSE_ROWS = [
    ("C01", "인공지능 첫걸음", "AI 기초", 1, 2, "인공지능이 무엇인지 사례로 알아보고 일상과 업무에서 쓰이는 방식을 배웁니다"),
    ("C02", "생성형 AI 안전하게 쓰기", "AI 기초", 1, 1, "생성형 AI에 질문하는 방법과 가상 자료만 입력하는 보안 수칙, 답변 검증 방법을 배웁니다"),
    ("C03", "머신러닝 맛보기", "AI 기초", 2, 3, "데이터로 규칙을 학습하는 머신러닝의 원리와 분류 예측 실습을 해 봅니다"),
    ("C04", "AI 프롬프트 작성 실습", "AI 기초", 2, 2, "목적에 맞는 프롬프트를 작성하고 결과를 비교하며 개선하는 실습을 합니다"),
    ("C05", "AI 윤리와 책임", "AI 기초", 2, 1, "AI 결과의 한계와 편향, 최종 판단은 사람이 한다는 원칙을 사례로 배웁니다"),
    ("C06", "엑셀 데이터 정리 기초", "데이터", 1, 2, "표 데이터를 정리하고 합계와 평균을 구하며 오류를 찾는 방법을 배웁니다"),
    ("C07", "데이터 시각화 입문", "데이터", 1, 2, "막대그래프와 꺾은선 그래프로 데이터를 읽기 쉽게 보여 주는 방법을 배웁니다"),
    ("C08", "파이썬 데이터 분석 기초", "데이터", 2, 4, "파이썬으로 표 데이터를 읽고 통계를 계산하며 그래프를 그리는 실습을 합니다"),
    ("C09", "통계로 읽는 부대 생활", "데이터", 2, 2, "평균과 분포, 상관관계로 생활 데이터를 해석하는 방법을 사례로 배웁니다"),
    ("C10", "데이터 품질 점검 실습", "데이터", 3, 3, "결측과 중복, 형식 오류를 찾아 고치고 데이터 신뢰도를 점검하는 방법을 실습합니다"),
    ("C11", "정보보안 기본 수칙", "보안", 1, 1, "비밀번호 관리와 외부 저장매체 사용 금지, 촬영 금지 등 보안 수칙을 배웁니다"),
    ("C12", "사이버 위협 알아보기", "보안", 1, 2, "악성 메일과 피싱 사례를 보고 의심스러운 연락에 대처하는 방법을 배웁니다"),
    ("C13", "계정과 비밀번호 관리", "보안", 2, 1, "안전한 비밀번호 만들기와 계정 점검 방법을 실습하며 보안 습관을 기릅니다"),
    ("C14", "보안 사고 대응 절차", "보안", 2, 2, "보안 사고가 의심될 때 보고하는 절차와 지휘계통 보고 요령을 사례로 배웁니다"),
    ("C15", "네트워크 보안 이해", "보안", 3, 4, "네트워크 구조와 접근 통제의 원리를 배우고 취약점 점검 개념을 이해합니다"),
    ("C16", "생활관 안전사고 예방", "안전", 1, 1, "전열기 사용과 결빙, 낙상 등 생활관에서 일어나는 안전사고 예방 수칙을 배웁니다"),
    ("C17", "응급처치 기본", "안전", 1, 2, "지혈과 심폐소생술 등 응급처치 기본 동작을 실습하고 보고 요령을 익힙니다"),
    ("C18", "화재 예방과 대피", "안전", 2, 2, "화재 원인과 초기 진화, 대피 경로 확인 방법을 훈련 사례로 배웁니다"),
    ("C19", "동절기 안전 관리", "안전", 2, 1, "한파와 결빙에 대비한 방한 수칙과 동상 예방, 안전 점검 방법을 배웁니다"),
    ("C20", "안전 리더 과정", "안전", 3, 3, "분대 단위 안전 점검표를 만들고 위험 요인을 찾아 개선하는 방법을 실습합니다"),
    ("C21", "분대장 리더십 기초", "리더십", 1, 2, "분대원을 이끄는 기본 자세와 지시, 보고 요령을 사례로 배웁니다"),
    ("C22", "팀 목표 세우기", "리더십", 1, 1, "팀의 목표를 정하고 역할을 나누며 진행 상황을 점검하는 방법을 배웁니다"),
    ("C23", "갈등 조정과 중재", "리더십", 2, 2, "생활관에서 생기는 갈등을 듣고 조정하는 대화 방법을 역할극으로 연습합니다"),
    ("C24", "멘토링과 후임 지도", "리더십", 2, 2, "후임을 지도하고 격려하는 멘토링 방법과 피드백 요령을 사례로 배웁니다"),
    ("C25", "지휘 판단과 의사결정", "리더십", 3, 3, "정보가 부족할 때 근거를 모아 결정하고 책임 있게 보고하는 방법을 사례로 배웁니다"),
    ("C26", "기초 체력 향상 프로그램", "체력", 1, 2, "팔굽혀펴기와 윗몸일으키기, 달리기 기록을 단계별로 올리는 체력 훈련을 합니다"),
    ("C27", "스트레칭과 부상 예방", "체력", 1, 1, "운동 전후 스트레칭과 올바른 자세로 근육 부상을 예방하는 방법을 배웁니다"),
    ("C28", "체력 측정 대비 훈련", "체력", 2, 2, "체력 측정 종목별 요령과 기록 향상 훈련 계획을 세우고 실천합니다"),
    ("C29", "영양과 회복 관리", "체력", 2, 1, "훈련 후 영양 섭취와 수면, 회복 관리로 체력을 유지하는 방법을 배웁니다"),
    ("C30", "전투 체력 심화 과정", "체력", 3, 3, "장거리 달리기와 근지구력을 높이는 심화 체력 훈련과 기록 관리 방법을 배웁니다"),
    ("C31", "컴퓨터활용능력 대비", "자격증 대비", 1, 4, "표 계산과 문서 작성 등 컴퓨터활용능력 시험 문제 풀이와 자격증 대비 요령을 익힙니다"),
    ("C32", "한국사능력 대비", "자격증 대비", 1, 3, "한국사 흐름을 정리하고 기출 문제를 풀며 자격증 시험을 대비합니다"),
    ("C33", "정보처리 기초 대비", "자격증 대비", 2, 4, "자료구조와 알고리즘, 데이터베이스 기초를 정리하고 자격증 시험 문제를 풉니다"),
    ("C34", "데이터 분석 자격 대비", "자격증 대비", 2, 4, "통계와 데이터 처리 개념을 정리하고 데이터 분석 자격증 시험을 대비합니다"),
    ("C35", "전기·정비 기능사 대비", "자격증 대비", 3, 4, "전기 회로와 정비 이론을 정리하고 실기 요령을 익혀 기능사 자격증을 대비합니다"),
    ("C36", "말하기와 발표 기초", "소통·정서", 1, 1, "핵심을 먼저 말하는 보고 방식과 발표 자세를 연습하며 소통 능력을 기릅니다"),
    ("C37", "스트레스 관리와 마음 건강", "소통·정서", 1, 2, "스트레스 신호를 알아차리고 상담 창구를 이용하는 방법 등 마음 건강 관리를 배웁니다"),
    ("C38", "경청과 공감 대화", "소통·정서", 2, 2, "상대의 말을 끝까지 듣고 공감하며 답하는 대화 방법을 짝 연습으로 익힙니다"),
    ("C39", "전역 후 진로 설계", "소통·정서", 2, 2, "전역 후 진로를 정하고 자기개발 계획과 자격증 준비 일정을 세우는 방법을 배웁니다"),
    ("C40", "부대 보고서 글쓰기", "소통·정서", 3, 3, "읽는 사람을 생각해 근거와 결론이 분명한 보고서와 공문을 쓰는 방법을 실습합니다"),
]
courses = pd.DataFrame(COURSE_ROWS, columns=["id", "title", "topic", "level", "hours", "desc"])
CID = list(courses["id"])
RANKS = ["이병", "일병", "상병", "병장"]

# 가상 장병 60명: 계급, 관심 주제 2개, 수강 이력과 평점(1~5)을 난수로 만든다
rng = np.random.default_rng(13)
item_pop = rng.lognormal(0, 0.6, len(courses))
rows, soldier_rows = [], []
for i in range(60):
    sid = f"S{i + 1:02d}"
    rank = RANKS[min(i // 15, 3)]
    interest = list(rng.choice(TOPICS, 2, replace=False))
    max_level = 1 + (RANKS.index(rank) >= 1) + (RANKS.index(rank) >= 3)
    w = np.array([item_pop[j] * (1 + 7 * (courses.topic[j] in interest)) * (1.0 if courses.level[j] <= max_level else 0.3)
                  for j in range(len(courses))])
    n_take = int(rng.integers(7, 14))
    picked = rng.choice(len(courses), n_take, replace=False, p=w / w.sum())
    for j in picked:
        score = 2.9 + 1.3 * (courses.topic[j] in interest) + rng.normal(0, 0.7)
        rows.append((sid, CID[j], int(np.clip(round(score), 1, 5))))
    soldier_rows.append((sid, rank, interest[0], interest[1]))
soldiers = pd.DataFrame(soldier_rows, columns=["id", "rank", "interest1", "interest2"])
ratings = pd.DataFrame(rows, columns=["soldier", "course", "rating"])
SID = list(soldiers["id"])

# 평가용 분리: 평점 4 이상인 수강 중 장병마다 최대 2개를 숨겨 두고(test), 나머지로 추천 모델을 만든다(train)
test_idx = []
for sid, g in ratings[ratings.rating >= 4].groupby("soldier"):
    test_idx += list(rng.choice(g.index, min(2, len(g)), replace=False))
test = ratings.loc[sorted(test_idx)].reset_index(drop=True)
train = ratings.drop(index=test_idx).reset_index(drop=True)
R = train.pivot(index="soldier", columns="course", values="rating").reindex(index=SID, columns=CID)
M = (~R.isna()).values.astype(float)        # 수강 여부(1/0)
RV = R.fillna(0).values                      # 평점 행렬(미수강은 0)
relevant = test.groupby("soldier")["course"].apply(set).to_dict()

# ---- 공통 함수: 점수 계산(인기도·콘텐츠·협업·행렬 분해), 상위 추천, 평가 ----
SIDX = {s: i for i, s in enumerate(SID)}
GLOBAL_MEAN = train["rating"].mean()
_stat = train.groupby("course")["rating"].agg(["mean", "count"]).reindex(CID).fillna({"mean": GLOBAL_MEAN, "count": 0})
POP_SCORE = ((_stat["count"] * _stat["mean"] + 5 * GLOBAL_MEAN) / (_stat["count"] + 5)).values  # 수강 수가 적으면 전체 평균 쪽으로 보정

_vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 3))
TFIDF = _vec.fit_transform(courses["title"] + " " + courses["topic"] + " " + courses["desc"])
ITEM_SIM = cosine_similarity(TFIDF)
USER_SIM = cosine_similarity(RV)
np.fill_diagonal(USER_SIM, 0)
_svd = TruncatedSVD(n_components=6, random_state=13)
_U = _svd.fit_transform(RV)
SVD_SCORE = _U @ _svd.components_


def pop_scores(sid):
    return POP_SCORE.copy()


def content_scores(sid):
    """평점 4 이상으로 수강한 콘텐츠의 설명과 비슷한 콘텐츠일수록 높은 점수"""
    r = RV[SIDX[sid]]
    w = np.where(r >= 4, r - 3, 0)
    if w.sum() == 0:
        return np.zeros(len(CID))
    return (ITEM_SIM * w).sum(axis=1) / w.sum()


def cf_scores(sid, n=10):
    """나와 평점 패턴이 비슷한 장병 n명이 높게 준 콘텐츠일수록 높은 점수"""
    i = SIDX[sid]
    nb = np.argsort(-USER_SIM[i])[:n]
    return (USER_SIM[i, nb][:, None] * RV[nb]).sum(axis=0) / USER_SIM[i, nb].sum()


def svd_scores(sid):
    return SVD_SCORE[SIDX[sid]].copy()


def norm(v, mask):
    """아직 안 들은 콘텐츠들 사이에서 0~1로 맞춘다"""
    x = v.copy()
    lo, hi = x[mask].min(), x[mask].max()
    return np.where(mask, (x - lo) / (hi - lo + 1e-9), 0)


def hybrid_scores(sid, w=(0.2, 0.4, 0.4)):
    unseen = M[SIDX[sid]] == 0
    return (w[0] * norm(pop_scores(sid), unseen) + w[1] * norm(content_scores(sid), unseen)
            + w[2] * norm(cf_scores(sid), unseen))


def top_k(scores, sid, k=5):
    s = np.where(M[SIDX[sid]] == 1, -1e9, scores)    # 이미 들은 콘텐츠는 제외
    return [CID[j] for j in np.argsort(-s)[:k]]


def evaluate(score_fn, k=5, sids=None):
    """장병마다 상위 k개를 추천해 정밀도(Precision@k), 재현율(Recall@k), 커버리지를 구한다."""
    prec, rec, shown = [], [], set()
    for sid, rel in relevant.items():
        if sids is not None and sid not in sids:
            continue
        rc = top_k(score_fn(sid), sid, k)
        hit = len(set(rc) & rel)
        prec.append(hit / k); rec.append(hit / len(rel)); shown |= set(rc)
    return np.mean(prec), np.mean(rec), len(shown) / len(CID)

# ---- 다양성: 같은 주제가 몰리지 않게 다시 정렬 ----
def diversify(scores, sid, k=5, max_per_topic=2):
    """점수 순으로 고르되, 같은 주제는 최대 max_per_topic개까지만 담는다."""
    s = np.where(M[SIDX[sid]] == 1, -1e9, scores)
    out, cnt = [], {}
    for j in np.argsort(-s):
        t = courses.topic[j]
        if cnt.get(t, 0) >= max_per_topic:
            continue
        out.append(CID[j]); cnt[t] = cnt.get(t, 0) + 1
        if len(out) == k:
            break
    return out


def topic_count(rc):
    return len({courses.topic[CID.index(c)] for c in rc})


def list_diversity(rc):
    """추천 목록 안 콘텐츠끼리의 설명이 얼마나 서로 다른지(1 - 평균 유사도)"""
    idx = [CID.index(c) for c in rc]
    sub = ITEM_SIM[np.ix_(idx, idx)]
    return 1 - (sub.sum() - len(idx)) / (len(idx) * (len(idx) - 1))


def report(name, rec_fn):
    topics, kinds, div, expo, hit, own = [], [], [], [], 0, []
    for sid in SID:
        rc = rec_fn(sid)
        me = soldiers[soldiers.id == sid].iloc[0]
        tp = [courses.topic[CID.index(c)] for c in rc]
        topics.append(len(set(tp))); div.append(list_diversity(rc))
        own.append(np.mean([t in (me["interest1"], me["interest2"]) for t in tp]))
        expo += rc
        if sid in relevant:
            hit += len(set(rc) & relevant[sid])
    cnt = pd.Series(expo).value_counts()
    top5_share = cnt.head(5).sum() / cnt.sum()
    print(f"{name}: Precision@5 {hit / (5 * len(relevant)):.3f} | 추천당 평균 주제 수 {np.mean(topics):.2f} | "
          f"목록 다양도 {np.mean(div):.2f} | 관심 주제 비율 {np.mean(own):.0%} | 노출 상위 5개 콘텐츠 점유 {top5_share:.0%} | 노출된 콘텐츠 {len(cnt)}개")
    return np.mean(topics), hit / (5 * len(relevant)), cnt


cont = lambda s: top_k(content_scores(s), s, 5)
base = lambda s: top_k(hybrid_scores(s), s, 5)
divr = lambda s: diversify(hybrid_scores(s), s, 5, 2)
tc, pc, _ = report("콘텐츠 기반 단독      ", cont)
t0, p0, c0 = report("기본 하이브리드      ", base)
t1, p1, c1 = report("하이브리드+주제 상한 2", divr)

print("\n[한 주제로 4개 이상 쏠린 장병 수]")
for name, fn in [("콘텐츠 기반 단독", cont), ("기본 하이브리드", base), ("주제 상한 적용", divr)]:
    bad = [s for s in SID if pd.Series([courses.topic[CID.index(c)] for c in fn(s)]).value_counts().iloc[0] >= 4]
    print(f"  {name}: {len(bad)}명 {bad[:6]}")
print("\n[가장 자주 노출된 콘텐츠 3개(기본 하이브리드)]:", [(courses.title[CID.index(k)], int(v)) for k, v in c0.head(3).items()])

print("\n[계급별 추천 난이도 평균(1 기초 ~ 3 심화)]")
for rk in RANKS:
    ids = soldiers.loc[soldiers["rank"] == rk, "id"]
    lv = np.mean([courses.level[CID.index(c)] for s in ids for c in base(s)])
    print(f"  {rk}: {lv:.2f}")

print(f"\n추천당 주제 수: 콘텐츠 단독 {tc:.2f} -> 하이브리드 {t0:.2f} -> 상한 적용 {t1:.2f}, 정확도: {pc:.3f} -> {p0:.3f} -> {p1:.3f}")
ok = t1 >= tc and p1 >= p0 * 0.8
print("다양성은 높이되 정확도 하락을 20% 이내로 유지:", "목표 달성" if ok else "목표 미달")
print("모델은 인기 있고 이미 좋아한 주제를 반복하는 경향이 있다. 다양성과 정확도의 균형은 사람이 정한다.")


---

## 🎯 미션 10. 추천 도우미 완성하기
**소요시간**: 약 2.0시간

### 📋 미션 배경 및 수행 지침
#### 미션 배경 및 당위성

실제 부대 실무 현장에서는 불명확한 정보 전달이나 수동 업무 처리로 인해 불필요한 행정 소요가 발생하고 주요 공지·규정 확인이 지연되는 문제가 발생하기 쉽습니다. 따라서 **전체 기능을 하나로 묶은 교육 콘텐츠 추천 도우미를 완성하고 자체 점검합니다.** 과업을 체계적으로 수행하여 현장 업무 효율성을 극대화하고 정보 누락으로 인한 혼선을 사전에 방지해야 합니다.

이 문제를 해결하기 위해 본 미션에서는 **기록 검증, 하이브리드 추천, 신규 전입 대응, 추천 이유, 다양성 조정을 하나의 도우미로 통합하고 추천 카드를 출력 수행하고 검증하는** 실습을 단계별로 수행합니다.

학습자는 실습가이드에 제시된 총 4개 세부 활동을 순차적으로 진행하며 핵심 산출물을 도출하고 검증 기준을 달성합니다.

**이번 미션 결과물:** `추천 도우미 코드, 자체 점검 결과, 사람 결정 안내와 검토용 프롬프트` 완성

### 🔍 평가 및 검증 기준 (PASS 조건)
#### 미션 판정

**평가 구분:** 필수

**핵심 평가 기준:** [필수] 통합·완성 — 도우미를 통합하고 자체 점검을 통과했으며 이수 결정은 사람이 한다는 안내를 명시했는가? (PASS / FAIL)

#### 미션 PASS 조건

- 요구 산출물(추천 도우미 코드, 자체 점검 결과, 사람 결정 안내와 검토용 프롬프트)의 완성도 충족


- 원문 대조 점검을 통한 사실 검증 및 정확성 확보


- 교육용 가상 데이터 활용 및 군사보안 수칙 엄수


#### 채점자 판정 메모

> 단순 결과물 생성 여부뿐만 아니라 AI 결과물의 오류를 직접 찾아내고 바로잡았는지 대조표를 확인한다.



In [ ]:
"""수강 기록 검증, 하이브리드 추천, 신규 전입 대응, 추천 이유, 다양성 조정을 하나로 묶은 '교육 콘텐츠 추천 도우미'를 완성하고 자체 점검한다."""
import time
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import TruncatedSVD

# ---------------------------------------------------------------------------
# [교육용가상자료] 가상 한빛대대 교육 콘텐츠 40개와 가상 장병 60명 (실제 부대·인원과 무관)
# ---------------------------------------------------------------------------
TOPICS = ["AI 기초", "데이터", "보안", "안전", "리더십", "체력", "자격증 대비", "소통·정서"]
LEVELS = {1: "기초", 2: "중급", 3: "심화"}
COURSE_ROWS = [
    ("C01", "인공지능 첫걸음", "AI 기초", 1, 2, "인공지능이 무엇인지 사례로 알아보고 일상과 업무에서 쓰이는 방식을 배웁니다"),
    ("C02", "생성형 AI 안전하게 쓰기", "AI 기초", 1, 1, "생성형 AI에 질문하는 방법과 가상 자료만 입력하는 보안 수칙, 답변 검증 방법을 배웁니다"),
    ("C03", "머신러닝 맛보기", "AI 기초", 2, 3, "데이터로 규칙을 학습하는 머신러닝의 원리와 분류 예측 실습을 해 봅니다"),
    ("C04", "AI 프롬프트 작성 실습", "AI 기초", 2, 2, "목적에 맞는 프롬프트를 작성하고 결과를 비교하며 개선하는 실습을 합니다"),
    ("C05", "AI 윤리와 책임", "AI 기초", 2, 1, "AI 결과의 한계와 편향, 최종 판단은 사람이 한다는 원칙을 사례로 배웁니다"),
    ("C06", "엑셀 데이터 정리 기초", "데이터", 1, 2, "표 데이터를 정리하고 합계와 평균을 구하며 오류를 찾는 방법을 배웁니다"),
    ("C07", "데이터 시각화 입문", "데이터", 1, 2, "막대그래프와 꺾은선 그래프로 데이터를 읽기 쉽게 보여 주는 방법을 배웁니다"),
    ("C08", "파이썬 데이터 분석 기초", "데이터", 2, 4, "파이썬으로 표 데이터를 읽고 통계를 계산하며 그래프를 그리는 실습을 합니다"),
    ("C09", "통계로 읽는 부대 생활", "데이터", 2, 2, "평균과 분포, 상관관계로 생활 데이터를 해석하는 방법을 사례로 배웁니다"),
    ("C10", "데이터 품질 점검 실습", "데이터", 3, 3, "결측과 중복, 형식 오류를 찾아 고치고 데이터 신뢰도를 점검하는 방법을 실습합니다"),
    ("C11", "정보보안 기본 수칙", "보안", 1, 1, "비밀번호 관리와 외부 저장매체 사용 금지, 촬영 금지 등 보안 수칙을 배웁니다"),
    ("C12", "사이버 위협 알아보기", "보안", 1, 2, "악성 메일과 피싱 사례를 보고 의심스러운 연락에 대처하는 방법을 배웁니다"),
    ("C13", "계정과 비밀번호 관리", "보안", 2, 1, "안전한 비밀번호 만들기와 계정 점검 방법을 실습하며 보안 습관을 기릅니다"),
    ("C14", "보안 사고 대응 절차", "보안", 2, 2, "보안 사고가 의심될 때 보고하는 절차와 지휘계통 보고 요령을 사례로 배웁니다"),
    ("C15", "네트워크 보안 이해", "보안", 3, 4, "네트워크 구조와 접근 통제의 원리를 배우고 취약점 점검 개념을 이해합니다"),
    ("C16", "생활관 안전사고 예방", "안전", 1, 1, "전열기 사용과 결빙, 낙상 등 생활관에서 일어나는 안전사고 예방 수칙을 배웁니다"),
    ("C17", "응급처치 기본", "안전", 1, 2, "지혈과 심폐소생술 등 응급처치 기본 동작을 실습하고 보고 요령을 익힙니다"),
    ("C18", "화재 예방과 대피", "안전", 2, 2, "화재 원인과 초기 진화, 대피 경로 확인 방법을 훈련 사례로 배웁니다"),
    ("C19", "동절기 안전 관리", "안전", 2, 1, "한파와 결빙에 대비한 방한 수칙과 동상 예방, 안전 점검 방법을 배웁니다"),
    ("C20", "안전 리더 과정", "안전", 3, 3, "분대 단위 안전 점검표를 만들고 위험 요인을 찾아 개선하는 방법을 실습합니다"),
    ("C21", "분대장 리더십 기초", "리더십", 1, 2, "분대원을 이끄는 기본 자세와 지시, 보고 요령을 사례로 배웁니다"),
    ("C22", "팀 목표 세우기", "리더십", 1, 1, "팀의 목표를 정하고 역할을 나누며 진행 상황을 점검하는 방법을 배웁니다"),
    ("C23", "갈등 조정과 중재", "리더십", 2, 2, "생활관에서 생기는 갈등을 듣고 조정하는 대화 방법을 역할극으로 연습합니다"),
    ("C24", "멘토링과 후임 지도", "리더십", 2, 2, "후임을 지도하고 격려하는 멘토링 방법과 피드백 요령을 사례로 배웁니다"),
    ("C25", "지휘 판단과 의사결정", "리더십", 3, 3, "정보가 부족할 때 근거를 모아 결정하고 책임 있게 보고하는 방법을 사례로 배웁니다"),
    ("C26", "기초 체력 향상 프로그램", "체력", 1, 2, "팔굽혀펴기와 윗몸일으키기, 달리기 기록을 단계별로 올리는 체력 훈련을 합니다"),
    ("C27", "스트레칭과 부상 예방", "체력", 1, 1, "운동 전후 스트레칭과 올바른 자세로 근육 부상을 예방하는 방법을 배웁니다"),
    ("C28", "체력 측정 대비 훈련", "체력", 2, 2, "체력 측정 종목별 요령과 기록 향상 훈련 계획을 세우고 실천합니다"),
    ("C29", "영양과 회복 관리", "체력", 2, 1, "훈련 후 영양 섭취와 수면, 회복 관리로 체력을 유지하는 방법을 배웁니다"),
    ("C30", "전투 체력 심화 과정", "체력", 3, 3, "장거리 달리기와 근지구력을 높이는 심화 체력 훈련과 기록 관리 방법을 배웁니다"),
    ("C31", "컴퓨터활용능력 대비", "자격증 대비", 1, 4, "표 계산과 문서 작성 등 컴퓨터활용능력 시험 문제 풀이와 자격증 대비 요령을 익힙니다"),
    ("C32", "한국사능력 대비", "자격증 대비", 1, 3, "한국사 흐름을 정리하고 기출 문제를 풀며 자격증 시험을 대비합니다"),
    ("C33", "정보처리 기초 대비", "자격증 대비", 2, 4, "자료구조와 알고리즘, 데이터베이스 기초를 정리하고 자격증 시험 문제를 풉니다"),
    ("C34", "데이터 분석 자격 대비", "자격증 대비", 2, 4, "통계와 데이터 처리 개념을 정리하고 데이터 분석 자격증 시험을 대비합니다"),
    ("C35", "전기·정비 기능사 대비", "자격증 대비", 3, 4, "전기 회로와 정비 이론을 정리하고 실기 요령을 익혀 기능사 자격증을 대비합니다"),
    ("C36", "말하기와 발표 기초", "소통·정서", 1, 1, "핵심을 먼저 말하는 보고 방식과 발표 자세를 연습하며 소통 능력을 기릅니다"),
    ("C37", "스트레스 관리와 마음 건강", "소통·정서", 1, 2, "스트레스 신호를 알아차리고 상담 창구를 이용하는 방법 등 마음 건강 관리를 배웁니다"),
    ("C38", "경청과 공감 대화", "소통·정서", 2, 2, "상대의 말을 끝까지 듣고 공감하며 답하는 대화 방법을 짝 연습으로 익힙니다"),
    ("C39", "전역 후 진로 설계", "소통·정서", 2, 2, "전역 후 진로를 정하고 자기개발 계획과 자격증 준비 일정을 세우는 방법을 배웁니다"),
    ("C40", "부대 보고서 글쓰기", "소통·정서", 3, 3, "읽는 사람을 생각해 근거와 결론이 분명한 보고서와 공문을 쓰는 방법을 실습합니다"),
]
courses = pd.DataFrame(COURSE_ROWS, columns=["id", "title", "topic", "level", "hours", "desc"])
CID = list(courses["id"])
RANKS = ["이병", "일병", "상병", "병장"]

# 가상 장병 60명: 계급, 관심 주제 2개, 수강 이력과 평점(1~5)을 난수로 만든다
rng = np.random.default_rng(13)
item_pop = rng.lognormal(0, 0.6, len(courses))
rows, soldier_rows = [], []
for i in range(60):
    sid = f"S{i + 1:02d}"
    rank = RANKS[min(i // 15, 3)]
    interest = list(rng.choice(TOPICS, 2, replace=False))
    max_level = 1 + (RANKS.index(rank) >= 1) + (RANKS.index(rank) >= 3)
    w = np.array([item_pop[j] * (1 + 7 * (courses.topic[j] in interest)) * (1.0 if courses.level[j] <= max_level else 0.3)
                  for j in range(len(courses))])
    n_take = int(rng.integers(7, 14))
    picked = rng.choice(len(courses), n_take, replace=False, p=w / w.sum())
    for j in picked:
        score = 2.9 + 1.3 * (courses.topic[j] in interest) + rng.normal(0, 0.7)
        rows.append((sid, CID[j], int(np.clip(round(score), 1, 5))))
    soldier_rows.append((sid, rank, interest[0], interest[1]))
soldiers = pd.DataFrame(soldier_rows, columns=["id", "rank", "interest1", "interest2"])
ratings = pd.DataFrame(rows, columns=["soldier", "course", "rating"])
SID = list(soldiers["id"])

# 평가용 분리: 평점 4 이상인 수강 중 장병마다 최대 2개를 숨겨 두고(test), 나머지로 추천 모델을 만든다(train)
test_idx = []
for sid, g in ratings[ratings.rating >= 4].groupby("soldier"):
    test_idx += list(rng.choice(g.index, min(2, len(g)), replace=False))
test = ratings.loc[sorted(test_idx)].reset_index(drop=True)
train = ratings.drop(index=test_idx).reset_index(drop=True)
R = train.pivot(index="soldier", columns="course", values="rating").reindex(index=SID, columns=CID)
M = (~R.isna()).values.astype(float)        # 수강 여부(1/0)
RV = R.fillna(0).values                      # 평점 행렬(미수강은 0)
relevant = test.groupby("soldier")["course"].apply(set).to_dict()

# ---- 공통 함수: 점수 계산(인기도·콘텐츠·협업·행렬 분해), 상위 추천, 평가 ----
SIDX = {s: i for i, s in enumerate(SID)}
GLOBAL_MEAN = train["rating"].mean()
_stat = train.groupby("course")["rating"].agg(["mean", "count"]).reindex(CID).fillna({"mean": GLOBAL_MEAN, "count": 0})
POP_SCORE = ((_stat["count"] * _stat["mean"] + 5 * GLOBAL_MEAN) / (_stat["count"] + 5)).values  # 수강 수가 적으면 전체 평균 쪽으로 보정

_vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 3))
TFIDF = _vec.fit_transform(courses["title"] + " " + courses["topic"] + " " + courses["desc"])
ITEM_SIM = cosine_similarity(TFIDF)
USER_SIM = cosine_similarity(RV)
np.fill_diagonal(USER_SIM, 0)
_svd = TruncatedSVD(n_components=6, random_state=13)
_U = _svd.fit_transform(RV)
SVD_SCORE = _U @ _svd.components_


def pop_scores(sid):
    return POP_SCORE.copy()


def content_scores(sid):
    """평점 4 이상으로 수강한 콘텐츠의 설명과 비슷한 콘텐츠일수록 높은 점수"""
    r = RV[SIDX[sid]]
    w = np.where(r >= 4, r - 3, 0)
    if w.sum() == 0:
        return np.zeros(len(CID))
    return (ITEM_SIM * w).sum(axis=1) / w.sum()


def cf_scores(sid, n=10):
    """나와 평점 패턴이 비슷한 장병 n명이 높게 준 콘텐츠일수록 높은 점수"""
    i = SIDX[sid]
    nb = np.argsort(-USER_SIM[i])[:n]
    return (USER_SIM[i, nb][:, None] * RV[nb]).sum(axis=0) / USER_SIM[i, nb].sum()


def svd_scores(sid):
    return SVD_SCORE[SIDX[sid]].copy()


def norm(v, mask):
    """아직 안 들은 콘텐츠들 사이에서 0~1로 맞춘다"""
    x = v.copy()
    lo, hi = x[mask].min(), x[mask].max()
    return np.where(mask, (x - lo) / (hi - lo + 1e-9), 0)


def hybrid_scores(sid, w=(0.2, 0.4, 0.4)):
    unseen = M[SIDX[sid]] == 0
    return (w[0] * norm(pop_scores(sid), unseen) + w[1] * norm(content_scores(sid), unseen)
            + w[2] * norm(cf_scores(sid), unseen))


def top_k(scores, sid, k=5):
    s = np.where(M[SIDX[sid]] == 1, -1e9, scores)    # 이미 들은 콘텐츠는 제외
    return [CID[j] for j in np.argsort(-s)[:k]]


def evaluate(score_fn, k=5, sids=None):
    """장병마다 상위 k개를 추천해 정밀도(Precision@k), 재현율(Recall@k), 커버리지를 구한다."""
    prec, rec, shown = [], [], set()
    for sid, rel in relevant.items():
        if sids is not None and sid not in sids:
            continue
        rc = top_k(score_fn(sid), sid, k)
        hit = len(set(rc) & rel)
        prec.append(hit / k); rec.append(hit / len(rel)); shown |= set(rc)
    return np.mean(prec), np.mean(rec), len(shown) / len(CID)

# ---- 신규 전입 장병(수강 이력 없음) 대응 ----
MAXLEVEL = {"이병": 1, "일병": 2, "상병": 2, "병장": 3}


def cold_scores(interests, rank):
    """이력이 없을 때는 관심 주제 일치(60%) + 인기도(30%) + 난이도 적합(10%)으로 점수를 매긴다."""
    topic_hit = courses["topic"].isin(interests).values.astype(float)
    level_fit = (courses["level"].values <= MAXLEVEL[rank]).astype(float)
    pop = (POP_SCORE - POP_SCORE.min()) / (POP_SCORE.max() - POP_SCORE.min())
    return 0.6 * topic_hit + 0.3 * pop + 0.1 * level_fit


def top_k_new(scores, k=5, exclude=()):
    s = scores.copy()
    for j, c in enumerate(CID):
        if c in exclude:
            s[j] = -1e9
    return [CID[j] for j in np.argsort(-s)[:k]]

# ---- 추천 이유 설명 ----
def explain(sid, cid, n=10):
    """추천 이유를 문장으로 만든다. 다른 장병의 이름은 밝히지 않고 인원수만 말한다."""
    i, j = SIDX[sid], CID.index(cid)
    c, me = courses.iloc[j], soldiers.iloc[i]
    why = []
    if c["topic"] in (me["interest1"], me["interest2"]):
        why.append(f"관심 주제('{c['topic']}')에 해당합니다")
    liked = [k for k in range(len(CID)) if RV[i, k] >= 4]
    if liked:
        k = max(liked, key=lambda x: ITEM_SIM[j, x])
        if ITEM_SIM[j, k] >= 0.15:
            why.append(f"높게 평가한 콘텐츠('{courses.title[k]}')와 설명이 비슷합니다(유사도 {ITEM_SIM[j, k]:.2f})")
    nb = np.argsort(-USER_SIM[i])[:n]
    fans = [x for x in nb if RV[x, j] >= 4]
    if fans:
        why.append(f"수강 패턴이 비슷한 장병 {n}명 중 {len(fans)}명이 높게 평가했습니다")
    if c["level"] <= MAXLEVEL[me["rank"]]:
        why.append(f"{me['rank']} 수준에 맞는 {LEVELS[c['level']]} 과정입니다")
    if not why:
        why.append("전체 장병에게 인기가 높은 과정입니다")
    return why

# ---- 다양성: 같은 주제가 몰리지 않게 다시 정렬 ----
def diversify(scores, sid, k=5, max_per_topic=2):
    """점수 순으로 고르되, 같은 주제는 최대 max_per_topic개까지만 담는다."""
    s = np.where(M[SIDX[sid]] == 1, -1e9, scores)
    out, cnt = [], {}
    for j in np.argsort(-s):
        t = courses.topic[j]
        if cnt.get(t, 0) >= max_per_topic:
            continue
        out.append(CID[j]); cnt[t] = cnt.get(t, 0) + 1
        if len(out) == k:
            break
    return out


def topic_count(rc):
    return len({courses.topic[CID.index(c)] for c in rc})


def list_diversity(rc):
    """추천 목록 안 콘텐츠끼리의 설명이 얼마나 서로 다른지(1 - 평균 유사도)"""
    idx = [CID.index(c) for c in rc]
    sub = ITEM_SIM[np.ix_(idx, idx)]
    return 1 - (sub.sum() - len(idx)) / (len(idx) * (len(idx) - 1))

NOTICE = "※ 이 추천은 참고용 제안입니다. 교육 이수 여부와 일정은 본인의 의사와 지휘관·교육 담당자의 확인으로 결정합니다."


class EduRecommender:
    def __init__(self, w=(0.1, 0.6, 0.3)):
        self.w = w

    def validate(self, rows):
        """새 수강 기록의 오류를 걸러 낸다."""
        ok, bad = [], []
        for sid, cid, r in rows:
            if sid in SIDX and cid in CID and 1 <= r <= 5:
                ok.append((sid, cid, r))
            else:
                bad.append((sid, cid, r))
        return ok, bad

    def recommend(self, sid=None, profile=None, k=5):
        """기존 장병(sid)은 하이브리드, 신규 전입(profile: 관심·계급)은 관심 주제 기반으로 추천한다."""
        if sid is not None and sid in SIDX and M[SIDX[sid]].sum() > 0:
            ids = diversify(hybrid_scores(sid, self.w), sid, k, 2)
            return [{"id": c, "title": courses.title[CID.index(c)], "hours": int(courses.hours[CID.index(c)]),
                     "why": explain(sid, c)} for c in ids]
        interests, rank = profile["관심"], profile["계급"]
        ids = top_k_new(cold_scores(interests, rank), k)
        out = []
        for c in ids:
            j = CID.index(c)
            why = [f"관심 주제('{courses.topic[j]}')에 해당합니다"] if courses.topic[j] in interests else ["전체 장병에게 인기가 높은 과정입니다"]
            why.append("수강 이력이 없어 관심 주제와 인기도로 추천했습니다")
            out.append({"id": c, "title": courses.title[j], "hours": int(courses.hours[j]), "why": why})
        return out

    def card(self, items):
        lines = [f"  {n}. {x['title']} ({x['hours']}시간) - {x['why'][0]}" for n, x in enumerate(items, 1)]
        return "\n".join(lines) + "\n  " + NOTICE


R_ = EduRecommender()
print("[기존 장병 S31 추천]")
print(R_.card(R_.recommend("S31")))
print("\n[신규 전입 장병(이력 없음) 추천]")
newp = {"관심": ["체력", "리더십"], "계급": "이병"}
print(R_.card(R_.recommend(profile=newp)))

t0 = time.time()
all_recs = {s: R_.recommend(s) for s in SID}
elapsed = time.time() - t0
ok_rows, bad_rows = R_.validate([("S01", "C01", 4), ("S99", "C01", 3), ("S02", "C99", 3), ("S03", "C02", 9)])
hit = sum(len({x["id"] for x in all_recs[s]} & relevant[s]) for s in relevant)
prec = hit / (5 * len(relevant))
pop_prec = evaluate(pop_scores)[0]
new_recs = R_.recommend(profile=newp)

checks = {
    "잘못된 수강 기록 3건 차단": len(ok_rows) == 1 and len(bad_rows) == 3,
    "이미 수강한 콘텐츠를 추천하지 않음": all(M[SIDX[s], CID.index(x["id"])] == 0 for s in SID for x in all_recs[s]),
    "장병 60명 모두 5개 추천": all(len(v) == 5 for v in all_recs.values()),
    "모든 추천에 이유 제시": all(x["why"] for v in all_recs.values() for x in v),
    "추천마다 서로 다른 주제 3개 이상": all(topic_count([x["id"] for x in v]) >= 3 for v in all_recs.values()),
    "신규 전입 장병도 5개 추천": len(new_recs) == 5 and all(x["why"] for x in new_recs),
    "Precision@5가 인기도의 2배 이상": prec >= 2 * pop_prec,
    "전체 처리 5초 이내": elapsed < 5,
    "다른 장병 ID 미노출": not any(s in " ".join(x["why"]) for v in all_recs.values() for x in v for s in SID),
}
print(f"\n통합 도우미 Precision@5 {prec:.3f} (인기도 {pop_prec:.3f}), 처리 시간 {elapsed:.2f}초")
for k, v in checks.items():
    print(f"[{'PASS' if v else 'FAIL'}] {k}")
print(f"자체 점검 {sum(checks.values())}/{len(checks)} 통과")

print("\n[생성형 AI로 추천 목록을 검토할 때 쓰는 프롬프트 예시]")
print("너는 장병 교육 담당자를 돕는 검토자다. 아래 가상 장병의 관심 주제·계급과 추천 목록을 보고, 한 주제에 쏠렸는지, "
      "난이도가 맞는지, 추천 이유가 근거 없이 단정적이지 않은지 지적하라. (가상 자료만 사용)")
print("\n" + NOTICE)
